# Analyze and standardize single cell RNA-seq references for each tissue type

Note: mapping each source annotation onto the shared category set requires choices for which reasonable alternatives exist.  
For instance:  
    * Merkel cells are placed in the Epithelial category, although they are sometimes considered specialized cells.  
    * Light zone follicular DC and Dark zone follicular DC are placed in the Myeloid category, although they are sometimes considered specialized cells.  
    * myometrial cells are placed in the Blood vessel category, although they are sometimes considered specialized cells.  
    * Tuft cells are placed in the Epithelial category, although they are sometimes considered specialized cells.
    * and likewise for the other borderline labels.

In [1]:
# Using process_slide conda environment
import scanpy as sc
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
from collections import defaultdict
from copy import deepcopy
from functools import lru_cache
from typing import Sequence
import scipy.sparse as sp
import anndata as ad

# Add the project root to sys.path so that 'src' can be imported
project_root = Path.cwd().parents[3]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.constants import get_full_path

In [2]:
# Folders with ref adatas
ref_folder_before = get_full_path('eval', 'eval_datasets', 'scRNAseq_references', 'scRNAseq_references_before')
ref_folder_after = get_full_path('eval', 'eval_datasets', 'scRNAseq_references', 'adatas')
ref_folder_after.mkdir(exist_ok=True)

# Mapping tissue -> ref adata filename
filename_dict = json.load(open('/path/to/ALHENA-paper/data/eval_scRNAseq_ref/ref_tissue_files.json'))
filename_dict

{'Breast': 'breast.h5ad',
 'BoneMarrow': 'bone_marrow.h5ad',
 'Cervix': 'cervix.h5ad',
 'Colon': 'intestine.h5ad',
 'Heart': 'heart.h5ad',
 'Kidney': 'kidney.h5ad',
 'Liver': 'liver_cell.h5ad',
 'Lung': 'lung.h5ad',
 'LymphNode': 'tonsil.h5ad',
 'Ovary': 'ovary.h5ad',
 'Pancreatic': 'pancreas_cell.h5ad',
 'Prostate': 'prostate.h5ad',
 'Skin': 'skin.h5ad',
 'Tonsil': 'tonsil.h5ad',
 'Uterus': 'uterus.h5ad'}

In [3]:
# ct_cat_id = ct_1
with open('/path/to/ALHENA-paper/data/annots/annot_dicts_ct_1/cat2idx.json') as f:
    label2cat = json.load(f)
label2cat.keys()

dict_keys(['T_NK', 'B_Plasma', 'Myeloid', 'Blood_vessel', 'Fibroblast_Myofibroblast', 'Epithelial', 'Specialized', 'Melanocyte', 'Unknown'])

In [4]:
HGNC_COMPLETE_SET_URL = (
    "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"
)


@lru_cache(maxsize=1)
def _load_hgnc_table(url: str = HGNC_COMPLETE_SET_URL) -> pd.DataFrame:
    hgnc = pd.read_csv(
        url,
        sep="\t",
        usecols=["hgnc_id", "symbol", "alias_symbol", "prev_symbol", "ensembl_gene_id", "status"],
        dtype=str,
        keep_default_na=False,
    )
    hgnc = hgnc[hgnc["status"].eq("Approved")].copy()
    hgnc = hgnc[hgnc["ensembl_gene_id"].ne("")].copy()
    hgnc = hgnc.drop_duplicates(subset=["hgnc_id"]).reset_index(drop=True)
    return hgnc


def _split_pipe(x: str) -> list[str]:
    if not x:
        return []
    return [s.strip() for s in x.split("|") if s.strip()]


def _build_symbol_maps(hgnc: pd.DataFrame):
    approved = {}
    alias_prev_targets = defaultdict(set)

    for symbol, alias_symbol, prev_symbol, ensembl_id in hgnc[
        ["symbol", "alias_symbol", "prev_symbol", "ensembl_gene_id"]
    ].itertuples(index=False, name=None):
        target = (symbol, ensembl_id)
        approved[symbol] = target

        for a in _split_pipe(alias_symbol):
            alias_prev_targets[a].add(target)
        for p in _split_pipe(prev_symbol):
            alias_prev_targets[p].add(target)

    alias_prev_unique = {}
    ambiguous = set()

    for term, targets in alias_prev_targets.items():
        # Important: never let an alias override an exact approved symbol
        if term in approved:
            continue
        if len(targets) == 1:
            alias_prev_unique[term] = next(iter(targets))
        else:
            ambiguous.add(term)

    return approved, alias_prev_unique, ambiguous


def _sum_total(x) -> float:
    return float(np.asarray(x.sum()).squeeze())


def _collapse_sparse_or_dense(X, keep_idx: np.ndarray, codes: np.ndarray, n_out: int):
    X = X[:, keep_idx]

    if sp.issparse(X):
        X = X.tocsr()
        mapper = sp.csr_matrix(
            (
                np.ones(len(keep_idx), dtype=np.int8),
                (np.arange(len(keep_idx)), codes),
            ),
            shape=(len(keep_idx), n_out),
        )
        return (X @ mapper).tocsr()

    X = np.asarray(X)
    out_T = np.zeros((n_out, X.shape[0]), dtype=X.dtype)
    np.add.at(out_T, codes, X.T)
    return out_T.T


def map_human_gene_symbols_to_ensembl(
    adata: ad.AnnData,
    gene_col: str = "gene_name",
    gene_id_col: str = "gene_ids",
    hgnc_url: str = HGNC_COMPLETE_SET_URL,
) -> tuple[ad.AnnData, dict, pd.DataFrame]:
    """
    Map human gene symbols in adata.var[gene_col] to Ensembl gene IDs.

    Rules:
    - exact approved HGNC symbols are kept
    - aliases / previous symbols are rescued only if uniquely mappable
    - ambiguous symbols are dropped
    - genes without confident Ensembl mapping are dropped
    - duplicated Ensembl IDs after mapping are merged by summing counts

    Returns
    -------
    adata_out : AnnData
        New AnnData with merged count matrix in .X
        and var columns:
            - gene_name : approved HGNC symbol
            - gene_ids  : Ensembl gene ID
    report : dict
        Mapping summary
    audit : pd.DataFrame
        Per-input-gene mapping audit
    """
    if gene_col not in adata.var.columns:
        raise KeyError(f"`adata.var` does not contain column {gene_col!r}")

    hgnc = _load_hgnc_table(hgnc_url)
    approved, alias_prev_unique, ambiguous = _build_symbol_maps(hgnc)

    input_symbols = adata.var[gene_col].astype("string").fillna("").str.strip()

    resolved_symbols = []
    resolved_ids = []
    status = []

    for s in input_symbols.tolist():
        if not s:
            resolved_symbols.append(pd.NA)
            resolved_ids.append(pd.NA)
            status.append("empty")
        elif s in approved:
            sym, ens = approved[s]
            resolved_symbols.append(sym)
            resolved_ids.append(ens)
            status.append("approved_symbol")
        elif s in alias_prev_unique:
            sym, ens = alias_prev_unique[s]
            resolved_symbols.append(sym)
            resolved_ids.append(ens)
            status.append("alias_or_previous_symbol")
        elif s in ambiguous:
            resolved_symbols.append(pd.NA)
            resolved_ids.append(pd.NA)
            status.append("ambiguous")
        else:
            resolved_symbols.append(pd.NA)
            resolved_ids.append(pd.NA)
            status.append("unmapped_or_no_ensembl")

    audit = pd.DataFrame(
        {
            "input_symbol": input_symbols.to_numpy(dtype=object),
            "resolved_symbol": resolved_symbols,
            gene_id_col: resolved_ids,
            "mapping_status": status,
        },
        index=adata.var_names.copy(),
    )

    mapped_mask = audit[gene_id_col].notna().to_numpy()
    keep_idx = np.flatnonzero(mapped_mask)

    if len(keep_idx) == 0:
        raise ValueError("No genes could be mapped uniquely to Ensembl IDs.")

    mapped = audit.iloc[keep_idx].copy()

    # Factorize Ensembl IDs in first-appearance order
    codes, unique_ids = pd.factorize(mapped[gene_id_col], sort=False)
    n_out = len(unique_ids)

    X_new = _collapse_sparse_or_dense(adata.X, keep_idx, codes, n_out)

    # Build new var: one row per Ensembl gene
    new_var = (
        mapped.groupby(gene_id_col, sort=False)
        .agg(
            **{
                gene_col: ("resolved_symbol", "first"),
                "n_input_features_collapsed": ("input_symbol", "size"),
                "collapsed_from": ("input_symbol", lambda x: "|".join(pd.unique(x.astype(str)))),
            }
        )
        .reset_index()
    )

    new_var.index = new_var[gene_id_col].astype(str)
    new_var.index.name = None

    duplicate_sizes = mapped[gene_id_col].value_counts()
    n_duplicate_ensembl_ids = int((duplicate_sizes > 1).sum())

    total_counts_before = _sum_total(adata.X)
    total_counts_retained = _sum_total(adata.X[:, keep_idx])

    report = {
        "n_input_genes": int(adata.n_vars),
        "n_mapped_genes": int(mapped.shape[0]),
        "pct_genes_mapped": 100.0 * float(mapped.shape[0]) / float(adata.n_vars),
        "n_output_genes_after_collapse": int(n_out),
        "n_ambiguous_genes": int((audit["mapping_status"] == "ambiguous").sum()),
        "n_unmapped_or_no_ensembl": int((audit["mapping_status"] == "unmapped_or_no_ensembl").sum()),
        "n_empty_gene_names": int((audit["mapping_status"] == "empty").sum()),
        "n_dropped_genes_total": int((~mapped_mask).sum()),
        "n_duplicate_ensembl_ids": n_duplicate_ensembl_ids,
        "n_duplicate_input_features_collapsed": int(mapped.shape[0] - n_out),
        "total_counts_before": total_counts_before,
        "total_counts_retained": total_counts_retained,
        "pct_total_counts_retained": (
            100.0 * total_counts_retained / total_counts_before
            if total_counts_before > 0 else np.nan
        ),
        "hgnc_url": hgnc_url,
    }

    adata_out = ad.AnnData(
        X=X_new,
        obs=adata.obs.copy(),
        var=new_var.copy(),
    )

    adata_out.uns["gene_id_mapping_report"] = report
    adata_out.uns["gene_id_mapping_audit"] = audit

    return adata_out, report, audit

## Breast

In [5]:
tissue = 'Breast'

In [6]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 174638 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'time_point', 'subject_id', 'age', 'gender', 'rna_source', 'other_metadata'
    var: 'gene'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'

In [7]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,anatomical_site,disease,platform,cell_sorting,time_point,subject_id,age,gender,rna_source,other_metadata
AAACCTGAGTGGGTTG-1--ERX6174477,3395.0,1525,CXCL14 mammary basal cell,ERX6174477,E-MTAB-10885,control,breast,NA,control,10x3',NA,NA,NA,33.0,F,cell,NA
AAACCTGCATCGGACC-1--ERX6174477,5321.0,1827,SFN mammary luminal progenitor,ERX6174477,E-MTAB-10885,control,breast,NA,control,10x3',NA,NA,NA,33.0,F,cell,NA
AAACCTGGTCTCGTTC-1--ERX6174477,2721.0,1251,CXCL14 mammary basal cell,ERX6174477,E-MTAB-10885,control,breast,NA,control,10x3',NA,NA,NA,33.0,F,cell,NA
AAACCTGTCCAGATCA-1--ERX6174477,2700.0,1284,Venous EC,ERX6174477,E-MTAB-10885,control,breast,NA,control,10x3',NA,NA,NA,33.0,F,cell,NA
AAACCTGTCCAGGGCT-1--ERX6174477,24794.0,4652,SFN mammary luminal progenitor,ERX6174477,E-MTAB-10885,control,breast,NA,control,10x3',NA,NA,NA,33.0,F,cell,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGAGCATTCAGGT-1--GSM5852274,89240.0,9376,KRT17 mammary luminal cell,GSM5852274,GSE195861,control,breast,NA,control,10x3',NA,NA,NA,NaN,NA,cell,NA
TTTGGTTAGACTCAAA-1--GSM5852274,29066.0,6852,KRT17 mammary luminal cell,GSM5852274,GSE195861,control,breast,NA,control,10x3',NA,NA,NA,NaN,NA,cell,NA
TTTGGTTAGGTCTGGA-1--GSM5852274,17592.0,4542,PIP mammary luminal cell,GSM5852274,GSE195861,control,breast,NA,control,10x3',NA,NA,NA,NaN,NA,cell,NA
TTTGTTGGTGCTAGCC-1--GSM5852274,10797.0,3691,PIP mammary luminal cell,GSM5852274,GSE195861,control,breast,NA,control,10x3',NA,NA,NA,NaN,NA,cell,NA


In [8]:
adata.var

,gene
MIR1302-2HG,MIR1302-2HG
FAM138A,FAM138A
OR4F5,OR4F5
AL627309.1,AL627309.1
AL627309.3,AL627309.3
...,...
AC233755.2,AC233755.2
AC233755.1,AC233755.1
AC240274.1,AC240274.1
AC213203.1,AC213203.1


In [9]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [10]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 174638 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'time_point', 'subject_id', 'age', 'gender', 'rna_source', 'other_metadata', 'tissue_HistoST'
    var: 'gene_name'


In [11]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
SFN mammary luminal progenitor              20927
KRT6B mammary basal cell                    18528
KRT17 mammary luminal cell                  18113
PIP mammary luminal cell                    16453
SAA2 mammary luminal progenitor             15450
APOD+PTGDS+ fibroblast                      10829
CFD+MGP+ fibroblast                          8036
Capillary EC                                 7414
CXCL14 mammary basal cell                    7391
Venous EC                                    6985
Secretoglobin mammary luminal progenitor     6950
SCGB3A1 mammary luminal progenitor           4919
Secretoglobin mammary luminal cell           3225
CCSER1 mammary basal cell                    3004
Vascular smooth muscle cell                  2655
CREB+MT1A+ vascular smooth muscle cell       2518
CXCL+ pericyte                               2196
Arterial EC                                  2121
CCL19/21 pericyte                            2062
Pericyte                       

In [12]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # Epithelial (mammary epi)
    # -------------------------
    "SFN mammary luminal progenitor": "Epithelial",
    "KRT6B mammary basal cell": "Epithelial",
    "KRT17 mammary luminal cell": "Epithelial",
    "PIP mammary luminal cell": "Epithelial",
    "SAA2 mammary luminal progenitor": "Epithelial",
    "CXCL14 mammary basal cell": "Epithelial",
    "Secretoglobin mammary luminal progenitor": "Epithelial",
    "SCGB3A1 mammary luminal progenitor": "Epithelial",
    "Secretoglobin mammary luminal cell": "Epithelial",
    "CCSER1 mammary basal cell": "Epithelial",
    "Cycling mammary luminal progenitor": "Epithelial",

    # -------------------------
    # Specialized mammary state
    # -------------------------
    # Kept separate because lactocytes are a specialized secretory mammary epithelial state
    "Lactocyte": "Specialized",

    # -------------------------
    # Fibroblast / stromal
    # -------------------------
    "APOD+PTGDS+ fibroblast": "Fibroblast_Myofibroblast",
    "CFD+MGP+ fibroblast": "Fibroblast_Myofibroblast",
    "CDH19+LAMA2+ fibroblast": "Fibroblast_Myofibroblast",
    "MFAP5+IGFBP6+ fibroblast": "Fibroblast_Myofibroblast",
    "GPC3+ fibroblast": "Fibroblast_Myofibroblast",
    "BNC2+ZFPM2+ fibroblast": "Fibroblast_Myofibroblast",

    # -------------------------
    # Blood vessel / mural / endothelial
    # -------------------------
    "Capillary EC": "Blood_vessel",
    "Venous EC": "Blood_vessel",
    "Arterial EC": "Blood_vessel",
    "Lymphatic EC": "Blood_vessel",
    "Pericyte": "Blood_vessel",
    "CXCL+ pericyte": "Blood_vessel",
    "CCL19/21 pericyte": "Blood_vessel",
    "Vascular smooth muscle cell": "Blood_vessel",
    "CREB+MT1A+ vascular smooth muscle cell": "Blood_vessel",

    # -------------------------
    # T / NK / innate lymphoid
    # -------------------------
    "GZMB CD8 T cell": "T_NK",
    "CD4 T cell": "T_NK",
    "GZMK CD8 T cell": "T_NK",
    "NK cell": "T_NK",
    "Treg cell": "T_NK",
    "ILC": "T_NK",

    # -------------------------
    # B / plasma
    # -------------------------
    "B cell": "B_Plasma",
    "Plasma cell": "B_Plasma",

    # -------------------------
    # Myeloid / APC
    # -------------------------
    "M1 macrophage": "Myeloid",
    "Macrophage": "Myeloid",
    "LYVE1 macrophage": "Myeloid",
    "Monocyte": "Myeloid",
    "Dendritic cell": "Myeloid",
    "pDC": "Myeloid",
    "Mast cell": "Myeloid",

    # -------------------------
    # Unknown
    # -------------------------
    "Unknown": "Unknown",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
Epithelial                  116220
Blood_vessel                 29009
Fibroblast_Myofibroblast     20151
T_NK                          4193
Myeloid                       2735
Specialized                   1807
B_Plasma                       523
Melanocyte                       0
Unknown                          0
Name: count, dtype: int64


In [13]:
new_adata

AnnData object with n_obs × n_vars = 174638 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'time_point', 'subject_id', 'age', 'gender', 'rna_source', 'other_metadata', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [14]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [15]:
report

{'n_input_genes': 33538,
 'n_mapped_genes': 23480,
 'pct_genes_mapped': 70.01013775418927,
 'n_output_genes_after_collapse': 23398,
 'n_ambiguous_genes': 16,
 'n_unmapped_or_no_ensembl': 10042,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 10058,
 'n_duplicate_ensembl_ids': 77,
 'n_duplicate_input_features_collapsed': 82,
 'total_counts_before': 1331473170.0,
 'total_counts_retained': 1321540306.0,
 'pct_total_counts_retained': 99.25399443084535,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [16]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
MIR1302-2HG,MIR1302-2HG,MIR1302-2HG,ENSG00000243485,approved_symbol
FAM138A,FAM138A,FAM138A,ENSG00000237613,approved_symbol
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
...,...,...,...,...
AC233755.2,AC233755.2,<NA>,<NA>,unmapped_or_no_ensembl
AC233755.1,AC233755.1,<NA>,<NA>,unmapped_or_no_ensembl
AC240274.1,AC240274.1,<NA>,<NA>,unmapped_or_no_ensembl
AC213203.1,AC213203.1,<NA>,<NA>,unmapped_or_no_ensembl


In [17]:
adata_ensembl

AnnData object with n_obs × n_vars = 174638 × 23398
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'time_point', 'subject_id', 'age', 'gender', 'rna_source', 'other_metadata', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [18]:
adata_ensembl.var

,gene_ids,gene_name,n_input_features_collapsed,collapsed_from
ENSG00000243485,ENSG00000243485,MIR1302-2HG,1,MIR1302-2HG
ENSG00000237613,ENSG00000237613,FAM138A,1,FAM138A
ENSG00000186092,ENSG00000186092,OR4F5,1,OR4F5
ENSG00000284733,ENSG00000284733,OR4F29,1,OR4F29
ENSG00000284662,ENSG00000284662,OR4F16,1,OR4F16
...,...,...,...,...
ENSG00000198886,ENSG00000198886,MT-ND4,1,MT-ND4
ENSG00000198786,ENSG00000198786,MT-ND5,1,MT-ND5
ENSG00000198695,ENSG00000198695,MT-ND6,1,MT-ND6
ENSG00000198727,ENSG00000198727,MT-CYB,1,MT-CYB


In [19]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## BoneMarrow

In [20]:
tissue = 'BoneMarrow'

In [21]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 421775 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source'
    var: 'gene'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'

In [22]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,disease,platform,age_group,cell_sorting,subject_id,age,gender,rna_source
AAACCTGAGGATTCGG-1--GSM5415745,4422.0,1352,Memory CD4 T cell,GSM5415745,GSE179346,control,bone marrow,control,10x3',NA,NA,NA,15.0,F,cell
AAACCTGCACGGTAAG-1--GSM5415745,4630.0,1391,Naive CD8 T cell,GSM5415745,GSE179346,control,bone marrow,control,10x3',NA,NA,NA,15.0,F,cell
AAACCTGGTCATCCCT-1--GSM5415745,4218.0,1403,Naive CD4 T cell,GSM5415745,GSE179346,control,bone marrow,control,10x3',NA,NA,NA,15.0,F,cell
AAACCTGTCACAGTAC-1--GSM5415745,4160.0,1447,Naive CD4 T cell,GSM5415745,GSE179346,control,bone marrow,control,10x3',NA,NA,NA,15.0,F,cell
AAACGGGTCCGCGGTA-1--GSM5415745,4089.0,1380,Naive CD8 T cell,GSM5415745,GSE179346,control,bone marrow,control,10x3',NA,NA,NA,15.0,F,cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTGTATAAACG-1--ERX4327600,654.0,513,Cycling proB cell,ERX4327600,E-MTAB-9389,control,bone marrow,control,10x3',17 post conception week,CD45 negative,NA,NaN,F,cell
TTTGGTTTCACCTCGT-1--ERX4327600,746.0,611,Cycling proB cell,ERX4327600,E-MTAB-9389,control,bone marrow,control,10x3',17 post conception week,CD45 negative,NA,NaN,F,cell
TTTGTCAAGTAGCGGT-1--ERX4327600,772.0,616,Myeloid pre-pDC,ERX4327600,E-MTAB-9389,control,bone marrow,control,10x3',17 post conception week,CD45 negative,NA,NaN,F,cell
TTTGTCACAATGGAAT-1--ERX4327600,1794.0,1193,Cycling preB cell,ERX4327600,E-MTAB-9389,control,bone marrow,control,10x3',17 post conception week,CD45 negative,NA,NaN,F,cell


In [23]:
adata.var

,gene
MIR1302-2HG,MIR1302-2HG
FAM138A,FAM138A
OR4F5,OR4F5
AL627309.1,AL627309.1
AL627309.3,AL627309.3
...,...
AC233755.2,AC233755.2
AC233755.1,AC233755.1
AC240274.1,AC240274.1
AC213203.1,AC213203.1


In [24]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [25]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 421775 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST'
    var: 'gene_name'


In [26]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
Naive CD4 T cell                      42874
MHCII low CD14 monocyte               37767
CD16 NK cell                          33214
Memory CD4 T cell                     29121
GZMB CD8 T cell                       19407
GZMK CD8 T cell                       14471
MHCII high CD14 monocyte              14236
PreB cell                             13636
ProB cell                             11507
Hematopoietic stem cell               11487
Cycling preB cell                     11133
Cycling proB cell                     10453
Naive CD8 T cell                       9641
Multipotent progenitor                 9593
Granulocyte-monocyte progenitor        9176
Promyelocyte                           9060
Memory B cell                          8749
Naive B cell                           8493
Early SOX4+ erythroblast               8476
Late hemoglobin+ erythroblast          8475
Treg cell                              7923
S100A+ preNeutrophil                   7867
Intermediate 

In [27]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK / ILC
    # -------------------------
    "Naive CD4 T cell": "T_NK",
    "Memory CD4 T cell": "T_NK",
    "GZMB CD8 T cell": "T_NK",
    "GZMK CD8 T cell": "T_NK",
    "Naive CD8 T cell": "T_NK",
    "Treg cell": "T_NK",
    "CD16 NK cell": "T_NK",
    "CD56 NK cell": "T_NK",
    "Tissue-resident NK cell": "T_NK",
    "MAIT cell": "T_NK",
    "Cycling T/NK cell": "T_NK",
    "INF-activated T cell": "T_NK",
    "Gamma delta T cell": "T_NK",
    "ILC": "T_NK",

    # -------------------------
    # B / Plasma / B progenitors
    # -------------------------
    "PreB cell": "B_Plasma",
    "ProB cell": "B_Plasma",
    "Cycling preB cell": "B_Plasma",
    "Cycling proB cell": "B_Plasma",
    "Memory B cell": "B_Plasma",
    "Naive B cell": "B_Plasma",
    "Plasma cell": "B_Plasma",

    # -------------------------
    # Myeloid (mono / neutro / DC / pDC / mast / macrophage)
    # -------------------------
    "MHCII low CD14 monocyte": "Myeloid",
    "MHCII high CD14 monocyte": "Myeloid",
    "CD16 monocyte": "Myeloid",
    "Macrophage": "Myeloid",
    "Granulocyte-monocyte progenitor": "Myeloid",
    "Common myeloid progenitor": "Myeloid",
    "Promyelocyte": "Myeloid",
    "Myelocyte": "Myeloid",
    "Metamyelocyte/Band neutrophil": "Myeloid",
    "S100A+ preNeutrophil": "Myeloid",
    "Cycling S100A+ preNeutrophil": "Myeloid",
    "cDC1": "Myeloid",
    "cDC2": "Myeloid",
    "Cycling cDC2": "Myeloid",
    "pDC": "Myeloid",
    "Common dendritic progenitor": "Myeloid",
    "Lymphoid pre-pDC": "Myeloid",
    "Myeloid pre-pDC": "Myeloid",
    "Mast cell": "Myeloid",

    # -------------------------
    # Fibroblast / stromal
    # -------------------------
    "Fibroblast": "Fibroblast_Myofibroblast",
    "Mesenchymal stromal cell": "Fibroblast_Myofibroblast",

    # -------------------------
    # Specialized
    # erythroid / megakaryocytic / osteoclast
    # -------------------------
    "Early SOX4+ erythroblast": "Specialized",
    "Intermediate EPCAM+ erythroblast": "Specialized",
    "Late hemoglobin+ erythroblast": "Specialized",
    "Fetal HBG+ erythrocyte": "Specialized",
    "Red blood cell": "Specialized",
    "Megakaryocyte–erythroid progenitor": "Specialized",
    "Megakaryocyte progenitor": "Specialized",
    "Megakaryocyte": "Specialized",
    "Osteoclast": "Specialized",

    # -------------------------
    # Upstream / ambiguous progenitors
    # -------------------------
    "Hematopoietic stem cell": "Unknown",
    "Multipotent progenitor": "Unknown",
    "Common lymphoid progenitor": "Unknown",

    # -------------------------
    # Optional safety
    # -------------------------
    "Unknown": "Unknown",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
T_NK                        172132
Myeloid                     120233
B_Plasma                     64431
Specialized                  36059
Unknown                      25706
Fibroblast_Myofibroblast      3214
Blood_vessel                     0
Epithelial                       0
Melanocyte                       0
Name: count, dtype: int64


In [28]:
new_adata

AnnData object with n_obs × n_vars = 421775 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [29]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [30]:
report

{'n_input_genes': 33538,
 'n_mapped_genes': 23480,
 'pct_genes_mapped': 70.01013775418927,
 'n_output_genes_after_collapse': 23398,
 'n_ambiguous_genes': 16,
 'n_unmapped_or_no_ensembl': 10042,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 10058,
 'n_duplicate_ensembl_ids': 77,
 'n_duplicate_input_features_collapsed': 82,
 'total_counts_before': 2261407488.0,
 'total_counts_retained': 2248701990.0,
 'pct_total_counts_retained': 99.43815972718669,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [31]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
MIR1302-2HG,MIR1302-2HG,MIR1302-2HG,ENSG00000243485,approved_symbol
FAM138A,FAM138A,FAM138A,ENSG00000237613,approved_symbol
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
...,...,...,...,...
AC233755.2,AC233755.2,<NA>,<NA>,unmapped_or_no_ensembl
AC233755.1,AC233755.1,<NA>,<NA>,unmapped_or_no_ensembl
AC240274.1,AC240274.1,<NA>,<NA>,unmapped_or_no_ensembl
AC213203.1,AC213203.1,<NA>,<NA>,unmapped_or_no_ensembl


In [32]:
adata_ensembl

AnnData object with n_obs × n_vars = 421775 × 23398
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [33]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Cervix

In [34]:
tissue = 'Cervix'
# https://cellxgene.cziscience.com/collections/38833785-fac5-48fd-944a-0f62a4c23ed1 contains transformed data, not raw counts
# No suitably annotated public reference was found for cervix; _1_scRNAseq_ref_cervix.ipynb builds one from GEO data
# Only the resulting cervix.h5ad is standardized here

In [35]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 77518 × 20615
    obs: 'gsm', 'sample', 'group', 'stage', 'replicate', 'histology', 'dataset', 'tissue', 'species', 'sample_id', 'cell_type', 'source_sample_id', 'concat_sample_id'
    var: 'gene_ids', 'feature_types'

In [36]:
adata.obs

,gsm,sample,group,stage,replicate,histology,dataset,tissue,species,sample_id,cell_type,source_sample_id,concat_sample_id
GSM6360680_N_HPV_NEG_1_AAACCCAAGGTGCGAT-1,GSM6360680,N_HPV_NEG_1,NO_HPV,normal,1,normal,GSE208653,cervix,Homo sapiens,GSM6360680_N_HPV_NEG_1,mast cells,GSM6360680_N_HPV_NEG_1,GSM6360680_N_HPV_NEG_1
GSM6360680_N_HPV_NEG_1_AAACCCACAATAACCC-1,GSM6360680,N_HPV_NEG_1,NO_HPV,normal,1,normal,GSE208653,cervix,Homo sapiens,GSM6360680_N_HPV_NEG_1,Langerhans cells / dendritic cells,GSM6360680_N_HPV_NEG_1,GSM6360680_N_HPV_NEG_1
GSM6360680_N_HPV_NEG_1_AAACCCACACATCCCT-1,GSM6360680,N_HPV_NEG_1,NO_HPV,normal,1,normal,GSE208653,cervix,Homo sapiens,GSM6360680_N_HPV_NEG_1,neutrophils,GSM6360680_N_HPV_NEG_1,GSM6360680_N_HPV_NEG_1
GSM6360680_N_HPV_NEG_1_AAACCCACACGGCCAT-1,GSM6360680,N_HPV_NEG_1,NO_HPV,normal,1,normal,GSE208653,cervix,Homo sapiens,GSM6360680_N_HPV_NEG_1,monocytes/macrophages,GSM6360680_N_HPV_NEG_1,GSM6360680_N_HPV_NEG_1
GSM6360680_N_HPV_NEG_1_AAACCCACAGTGTGCC-1,GSM6360680,N_HPV_NEG_1,NO_HPV,normal,1,normal,GSE208653,cervix,Homo sapiens,GSM6360680_N_HPV_NEG_1,cytotoxic CD8+ T cells,GSM6360680_N_HPV_NEG_1,GSM6360680_N_HPV_NEG_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM6360688_ADC_6_TTTGTTGGTCCCTGAG-1,GSM6360688,ADC_6,CA_HPV,cancer,3,adenocarcinoma,GSE208653,cervix,Homo sapiens,GSM6360688_ADC_6,CD4+ T cells,GSM6360688_ADC_6,GSM6360688_ADC_6
GSM6360688_ADC_6_TTTGTTGGTGGAACAC-1,GSM6360688,ADC_6,CA_HPV,cancer,3,adenocarcinoma,GSE208653,cervix,Homo sapiens,GSM6360688_ADC_6,Neutrophils,GSM6360688_ADC_6,GSM6360688_ADC_6
GSM6360688_ADC_6_TTTGTTGTCAGAGCGA-1,GSM6360688,ADC_6,CA_HPV,cancer,3,adenocarcinoma,GSE208653,cervix,Homo sapiens,GSM6360688_ADC_6,Mast cells,GSM6360688_ADC_6,GSM6360688_ADC_6
GSM6360688_ADC_6_TTTGTTGTCTCCCATG-1,GSM6360688,ADC_6,CA_HPV,cancer,3,adenocarcinoma,GSE208653,cervix,Homo sapiens,GSM6360688_ADC_6,Epithelial tumor cells,GSM6360688_ADC_6,GSM6360688_ADC_6


In [37]:
adata.var

,gene_ids,feature_types
OR4F5,ENSG00000186092,Gene Expression
OR4F29,ENSG00000284733,Gene Expression
OR4F16,ENSG00000284662,Gene Expression
SAMD11,ENSG00000187634,Gene Expression
NOC2L,ENSG00000188976,Gene Expression
...,...,...
AC233755.2,ENSG00000277856,Gene Expression
AC233755.1,ENSG00000275063,Gene Expression
AC240274.1,ENSG00000271254,Gene Expression
AC213203.4,ENSG00000277475,Gene Expression


In [38]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [39]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={"cell_type": "original_cell_type"})
new_adata.obs["tissue_HistoST"] = tissue
new_adata.obs["total_count"] = np.asarray(new_adata.X.sum(axis=1)).ravel()

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .cat.add_categories(["Unknown"])
    .fillna("Unknown")
)

# Add a column "gene_name" in var
new_adata.var["gene_name"] = new_adata.var_names.to_numpy()

print(new_adata)

AnnData object with n_obs × n_vars = 77518 × 20615
    obs: 'gsm', 'sample', 'group', 'stage', 'replicate', 'histology', 'dataset', 'tissue', 'species', 'sample_id', 'original_cell_type', 'source_sample_id', 'concat_sample_id', 'tissue_HistoST', 'total_count'
    var: 'gene_ids', 'feature_types', 'gene_name'


In [40]:
new_adata.var

,gene_ids,feature_types,gene_name
OR4F5,ENSG00000186092,Gene Expression,OR4F5
OR4F29,ENSG00000284733,Gene Expression,OR4F29
OR4F16,ENSG00000284662,Gene Expression,OR4F16
SAMD11,ENSG00000187634,Gene Expression,SAMD11
NOC2L,ENSG00000188976,Gene Expression,NOC2L
...,...,...,...
AC233755.2,ENSG00000277856,Gene Expression,AC233755.2
AC233755.1,ENSG00000275063,Gene Expression,AC233755.1
AC240274.1,ENSG00000271254,Gene Expression,AC240274.1
AC213203.4,ENSG00000277475,Gene Expression,AC213203.4


In [41]:
for cell_type, count in new_adata.obs.original_cell_type.value_counts(dropna=False).items():
    print(f"{cell_type}: {count} cells")

Neutrophils: 5182 cells
CD4+ T cells: 4441 cells
Cytotoxic T/NK cells: 3392 cells
T cells (IL7R+; likely CD4 memory/activated): 3373 cells
Squamous epithelial cells: 3247 cells
plasma cells: 2409 cells
epithelial cells: 2176 cells
Macrophages: 2163 cells
T cells: 2124 cells
secretory endocervical epithelial cells: 2039 cells
T cells (likely cytotoxic alpha-beta T cells): 1991 cells
neutrophils: 1899 cells
Cytotoxic T cells (CD8-like): 1772 cells
squamous epithelial cells: 1759 cells
CD8 T cells: 1688 cells
Cytotoxic T cells: 1521 cells
Endocervical epithelial cells (columnar/secretory): 1478 cells
NK cells: 1465 cells
Regulatory T cells: 1416 cells
endocervical / glandular epithelial cells: 1377 cells
Fibroblasts (matrix/homeostatic): 1275 cells
cytotoxic CD8+ T cells: 1237 cells
CD4 T cells: 1219 cells
fibroblasts: 1196 cells
B cells: 1121 cells
Fibroblasts: 1077 cells
Epithelial cells: 1037 cells
Plasma cells: 1028 cells
Fibroblasts (activated/ECM-remodeling): 966 cells
Endothelial c

In [42]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK / innate lymphoid
    # -------------------------
    "CD4+ T cells": "T_NK",
    "Cytotoxic T/NK cells": "T_NK",
    "T cells (IL7R+; likely CD4 memory/activated)": "T_NK",
    "T cells": "T_NK",
    "T cells (likely cytotoxic alpha-beta T cells)": "T_NK",
    "Cytotoxic T cells (CD8-like)": "T_NK",
    "CD8 T cells": "T_NK",
    "Cytotoxic T cells": "T_NK",
    "NK cells": "T_NK",
    "Regulatory T cells": "T_NK",
    "cytotoxic CD8+ T cells": "T_NK",
    "CD4 T cells": "T_NK",
    "CD8+ cytotoxic T cells": "T_NK",
    "gamma-delta T cells": "T_NK",
    "CD8+ / cytotoxic T cells": "T_NK",
    "γδ T cells": "T_NK",
    "Activated/exhausted T cells": "T_NK",
    "ILC3-like innate lymphoid cells": "T_NK",
    "Regulatory T cells (Treg)": "T_NK",
    "NK / cytotoxic lymphocytes": "T_NK",
    "Gamma-delta T cells": "T_NK",
    "gamma-delta T / NK-like cytotoxic lymphocytes": "T_NK",
    "T / NK cells": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "plasma cells": "B_Plasma",
    "B cells": "B_Plasma",
    "Plasma cells": "B_Plasma",
    "naive B cells": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "Neutrophils": "Myeloid",
    "Macrophages": "Myeloid",
    "neutrophils": "Myeloid",
    "Mast cells": "Myeloid",
    "mast cells": "Myeloid",
    "C1Q+ macrophages": "Myeloid",
    "C1QC+ macrophages": "Myeloid",
    "Monocytes/macrophages": "Myeloid",
    "Myeloid cells": "Myeloid",
    "monocytes/macrophages": "Myeloid",
    "Macrophages/monocytes": "Myeloid",
    "Plasmacytoid dendritic cells": "Myeloid",
    "macrophages / monocytes": "Myeloid",
    "Langerhans cells / dendritic cells": "Myeloid",
    "FCN1+ monocytes/macrophages": "Myeloid",
    "Inflammatory myeloid cells": "Myeloid",
    "Activated neutrophils": "Myeloid",
    "Conventional dendritic cells": "Myeloid",

    # -------------------------
    # Blood vessel / perivascular
    # -------------------------
    "Endothelial cells": "Blood_vessel",
    "endothelial cells": "Blood_vessel",
    "Pericytes": "Blood_vessel",
    "pericytes": "Blood_vessel",
    "pericytes / vascular smooth muscle-like cells": "Blood_vessel",
    "mural cells (pericytes / vascular smooth muscle cells)": "Blood_vessel",
    "Lymphatic endothelial cells": "Blood_vessel",

    # -------------------------
    # Fibroblast / myofibroblast / stromal
    # -------------------------
    "Fibroblasts (matrix/homeostatic)": "Fibroblast_Myofibroblast",
    "fibroblasts": "Fibroblast_Myofibroblast",
    "Fibroblasts": "Fibroblast_Myofibroblast",
    "Fibroblasts (activated/ECM-remodeling)": "Fibroblast_Myofibroblast",
    "Fibroblasts / CAFs": "Fibroblast_Myofibroblast",
    "stromal fibroblasts": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "Squamous epithelial cells": "Epithelial",
    "epithelial cells": "Epithelial",
    "secretory endocervical epithelial cells": "Epithelial",
    "squamous epithelial cells": "Epithelial",
    "Endocervical epithelial cells (columnar/secretory)": "Epithelial",
    "endocervical / glandular epithelial cells": "Epithelial",
    "Epithelial cells": "Epithelial",
    "Malignant epithelial cells": "Epithelial",
    "Epithelial cells (differentiated squamous-like)": "Epithelial",
    "glandular/secretory epithelial cells": "Epithelial",
    "cycling/proliferating epithelial cells": "Epithelial",
    "basal/reserve epithelial cells": "Epithelial",
    "Ciliated epithelial cells": "Epithelial",
    "ciliated epithelial cells": "Epithelial",
    "Glandular epithelial cells": "Epithelial",
    "secretory glandular epithelial cells": "Epithelial",
    "Epithelial tumor cells (glandular/secretory-like)": "Epithelial",
    "epithelial cells (unclassified/rare)": "Epithelial",
    "Epithelial tumor cells": "Epithelial",
    "Epithelial tumor cells (secretory-like)": "Epithelial",
    "Epithelial cells (secretory-like)": "Epithelial",
    "ciliated endocervical epithelial cells": "Epithelial",
    "Secretory epithelial cells": "Epithelial",
    "Malignant squamous epithelial cells": "Epithelial",
    "Keratinizing squamous epithelial cells": "Epithelial",
    "Epithelial cells (basal/squamous-like)": "Epithelial",
    "Tuft cells": "Epithelial",

    # -------------------------
    # Unknown (lineage not resolvable)
    # -------------------------
    "Cycling/proliferating cells": "Unknown",
    "cycling/proliferating cells": "Unknown",
    "Cycling cells": "Unknown",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
T_NK                        30608
Epithelial                  18890
Myeloid                     14871
Fibroblast_Myofibroblast     4896
B_Plasma                     4596
Blood_vessel                 2312
Unknown                      1345
Specialized                     0
Melanocyte                      0
Name: count, dtype: int64


In [43]:
new_adata

AnnData object with n_obs × n_vars = 77518 × 20615
    obs: 'gsm', 'sample', 'group', 'stage', 'replicate', 'histology', 'dataset', 'tissue', 'species', 'sample_id', 'original_cell_type', 'source_sample_id', 'concat_sample_id', 'tissue_HistoST', 'total_count', 'ct_1_cell_type'
    var: 'gene_ids', 'feature_types', 'gene_name'

In [44]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [45]:
report

{'n_input_genes': 20615,
 'n_mapped_genes': 19977,
 'pct_genes_mapped': 96.90516614115936,
 'n_output_genes_after_collapse': 19974,
 'n_ambiguous_genes': 4,
 'n_unmapped_or_no_ensembl': 634,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 638,
 'n_duplicate_ensembl_ids': 3,
 'n_duplicate_input_features_collapsed': 3,
 'total_counts_before': 718513920.0,
 'total_counts_retained': 718124544.0,
 'pct_total_counts_retained': 99.9458081480175,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [46]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
OR4F29,OR4F29,OR4F29,ENSG00000284733,approved_symbol
OR4F16,OR4F16,OR4F16,ENSG00000284662,approved_symbol
SAMD11,SAMD11,SAMD11,ENSG00000187634,approved_symbol
NOC2L,NOC2L,NOC2L,ENSG00000188976,approved_symbol
...,...,...,...,...
AC233755.2,AC233755.2,<NA>,<NA>,unmapped_or_no_ensembl
AC233755.1,AC233755.1,<NA>,<NA>,unmapped_or_no_ensembl
AC240274.1,AC240274.1,<NA>,<NA>,unmapped_or_no_ensembl
AC213203.4,AC213203.4,<NA>,<NA>,unmapped_or_no_ensembl


In [47]:
adata_ensembl

AnnData object with n_obs × n_vars = 77518 × 19974
    obs: 'gsm', 'sample', 'group', 'stage', 'replicate', 'histology', 'dataset', 'tissue', 'species', 'sample_id', 'original_cell_type', 'source_sample_id', 'concat_sample_id', 'tissue_HistoST', 'total_count', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [48]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Colon

In [49]:
tissue = 'Colon'

In [50]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 414406 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'treatment', 'subject_id', 'age', 'gender', 'rna_source', 'other_metadata'
    var: 'gene'
    obsm: 'X_pca', 'X_umap'

In [51]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,anatomical_site,disease,platform,age_group,cell_sorting,treatment,subject_id,age,gender,rna_source,other_metadata
AAACCTGCACCGATAT-1--ERX3382888,4837.0,1433,Naive CD4 T cell,ERX3382888,E-MTAB-8007,control,intestine,caecum,control,10x3',60 to 64 years old,"CD4-positive, alpha-beta T cell",NA,290b,NaN,F,cell,NA
AAACCTGCACCTCGTT-1--ERX3382888,4585.0,1419,Treg cell,ERX3382888,E-MTAB-8007,control,intestine,caecum,control,10x3',60 to 64 years old,"CD4-positive, alpha-beta T cell",NA,290b,NaN,F,cell,NA
AAACCTGTCTCACATT-1--ERX3382888,3460.0,1168,Naive CD4 T cell,ERX3382888,E-MTAB-8007,control,intestine,caecum,control,10x3',60 to 64 years old,"CD4-positive, alpha-beta T cell",NA,290b,NaN,F,cell,NA
AAACCTGTCTTGCATT-1--ERX3382888,16600.0,1950,Plasma cell,ERX3382888,E-MTAB-8007,control,intestine,caecum,control,10x3',60 to 64 years old,"CD4-positive, alpha-beta T cell",NA,290b,NaN,F,cell,NA
AAACGGGCAGCGTTCG-1--ERX3382888,2674.0,961,Naive CD4 T cell,ERX3382888,E-MTAB-8007,control,intestine,caecum,control,10x3',60 to 64 years old,"CD4-positive, alpha-beta T cell",NA,290b,NaN,F,cell,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGGTTCGGTCG-1--GSM7805180,4704.0,1654,Naive CD4 T cell,GSM7805180,GSE244031,control,intestine,appendix,control,10x3',NA,NA,NA,NA,58.0,F,cell,{ethnicity: non-Hispanic white}
TTTGTTGTCGACATCA-1--GSM7805180,8346.0,2743,Naive B cell,GSM7805180,GSE244031,control,intestine,appendix,control,10x3',NA,NA,NA,NA,58.0,F,cell,{ethnicity: non-Hispanic white}
TTTGTTGTCGCATAGT-1--GSM7805180,6453.0,1905,Naive CD4 T cell,GSM7805180,GSE244031,control,intestine,appendix,control,10x3',NA,NA,NA,NA,58.0,F,cell,{ethnicity: non-Hispanic white}
TTTGTTGTCGCCTAGG-1--GSM7805180,6331.0,2512,Naive B cell,GSM7805180,GSE244031,control,intestine,appendix,control,10x3',NA,NA,NA,NA,58.0,F,cell,{ethnicity: non-Hispanic white}


In [52]:
adata.var

,gene
MIR1302-2HG,MIR1302-2HG
FAM138A,FAM138A
OR4F5,OR4F5
AL627309.1,AL627309.1
AL627309.3,AL627309.3
...,...
AC233755.2,AC233755.2
AC233755.1,AC233755.1
AC240274.1,AC240274.1
AC213203.1,AC213203.1


In [53]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [54]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 414406 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'treatment', 'subject_id', 'age', 'gender', 'rna_source', 'other_metadata', 'tissue_HistoST'
    var: 'gene_name'


In [55]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
Plasma cell                    52198
Memory CD4 T cell              49888
Memory B cell                  36232
Fibroblast                     34714
Activated CD8 T cell           32079
Naive CD4 T cell               26225
Naive B cell                   25418
GZMK CD8 T cell                13815
Gamma delta T cell             13127
Treg cell                      12681
Colonocyte                     11436
Germinal center B cell          9777
Tfh cell                        8075
CD14 monocyte                   7302
Transit-amplifying cell         7212
Neutrophil                      6505
Venous EC                       6196
CD16 NK cell                    6077
Mast cell                       4674
cDC2                            4257
Vascular smooth muscle cell     4087
Goblet cell                     4027
CD56 NK cell                    3245
CXCR6 CD4 T cell                3176
MAIT cell                       3107
ILC                             2981
Enterocyte         

In [56]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK / innate lymphoid
    # -------------------------
    "Memory CD4 T cell": "T_NK",
    "Activated CD8 T cell": "T_NK",
    "Naive CD4 T cell": "T_NK",
    "GZMK CD8 T cell": "T_NK",
    "Gamma delta T cell": "T_NK",
    "Treg cell": "T_NK",
    "Tfh cell": "T_NK",
    "CD16 NK cell": "T_NK",
    "CD56 NK cell": "T_NK",
    "CXCR6 CD4 T cell": "T_NK",
    "MAIT cell": "T_NK",
    "ILC": "T_NK",
    "GZMB CD8 T cell": "T_NK",
    "Naive CD8 T cell": "T_NK",
    "Cycling T/NK cell": "T_NK",
    "IL22 CD4 T cell": "T_NK",
    "CXCL13 exhausted CD8 T cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "Plasma cell": "B_Plasma",
    "Memory B cell": "B_Plasma",
    "Naive B cell": "B_Plasma",
    "Germinal center B cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "CD14 monocyte": "Myeloid",
    "CD16 monocyte": "Myeloid",
    "LYVE1 macrophage": "Myeloid",
    "SPP1 macrophage": "Myeloid",
    "Neutrophil": "Myeloid",
    "Mast cell": "Myeloid",
    "Basophil": "Myeloid",
    "cDC1": "Myeloid",
    "cDC2": "Myeloid",
    "mregDC": "Myeloid",
    "pDC": "Myeloid",

    # -------------------------
    # Blood vessel
    # -------------------------
    "Venous EC": "Blood_vessel",
    "Arterial EC": "Blood_vessel",
    "Capillary EC": "Blood_vessel",
    "Lymphatic EC": "Blood_vessel",
    "Pericyte": "Blood_vessel",
    "Vascular smooth muscle cell": "Blood_vessel",
    "EndoMT": "Blood_vessel",

    # -------------------------
    # Fibroblast / Myofibroblast
    # -------------------------
    "Fibroblast": "Fibroblast_Myofibroblast",
    "Myofibroblast": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "Colonocyte": "Epithelial",
    "Transit-amplifying cell": "Epithelial",
    "Goblet cell": "Epithelial",
    "Enterocyte": "Epithelial",
    "BEST4 epithelial cell": "Epithelial",
    "Intestinal stem cell": "Epithelial",
    "Paneth cell": "Epithelial",
    "Tuft cell": "Epithelial",
    "Enteroendocrine cell": "Epithelial",

    # -------------------------
    # Specialized
    # -------------------------
    "Schwann cell": "Specialized",
    "Mesothelial cell": "Specialized",

    # -------------------------
    # Explicit fallback labels
    # -------------------------
    "Unknown": "Unknown",
    "ToRemove": "Unknown",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
T_NK                        181258
B_Plasma                    123625
Fibroblast_Myofibroblast     35286
Epithelial                   29870
Myeloid                      26988
Blood_vessel                 15085
Specialized                   2294
Melanocyte                       0
Unknown                          0
Name: count, dtype: int64


In [57]:
new_adata

AnnData object with n_obs × n_vars = 414406 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'treatment', 'subject_id', 'age', 'gender', 'rna_source', 'other_metadata', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [58]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [59]:
report

{'n_input_genes': 33538,
 'n_mapped_genes': 23480,
 'pct_genes_mapped': 70.01013775418927,
 'n_output_genes_after_collapse': 23398,
 'n_ambiguous_genes': 16,
 'n_unmapped_or_no_ensembl': 10042,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 10058,
 'n_duplicate_ensembl_ids': 77,
 'n_duplicate_input_features_collapsed': 82,
 'total_counts_before': 2743425728.0,
 'total_counts_retained': 2720719745.0,
 'pct_total_counts_retained': 99.172349272362,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [60]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
MIR1302-2HG,MIR1302-2HG,MIR1302-2HG,ENSG00000243485,approved_symbol
FAM138A,FAM138A,FAM138A,ENSG00000237613,approved_symbol
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
...,...,...,...,...
AC233755.2,AC233755.2,<NA>,<NA>,unmapped_or_no_ensembl
AC233755.1,AC233755.1,<NA>,<NA>,unmapped_or_no_ensembl
AC240274.1,AC240274.1,<NA>,<NA>,unmapped_or_no_ensembl
AC213203.1,AC213203.1,<NA>,<NA>,unmapped_or_no_ensembl


In [61]:
adata_ensembl

AnnData object with n_obs × n_vars = 414406 × 23398
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'treatment', 'subject_id', 'age', 'gender', 'rna_source', 'other_metadata', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [62]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Heart

In [63]:
tissue = 'Heart'

In [64]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

$HOME/miniforge3/envs/process_slide/lib/python3.12/site-packages/anndata/_io/h5ad.py:266: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  return AnnData(**{


AnnData object with n_obs × n_vars = 624358 × 33234
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source'
    var: 'gene'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances'

In [65]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,anatomical_site,disease,platform,age_group,cell_sorting,subject_id,age,gender,rna_source
AAACCTGAGAGTACCG-1--GSM5572792,3742.0,1923,Pericyte,GSM5572792,GSE183852,control,heart,Transmural LV Apex,control,10x5',NA,NA,NA,NaN,NA,nucleus
AAACCTGAGCTGCCCA-1--GSM5572792,1136.0,702,Capillary EC,GSM5572792,GSE183852,control,heart,Transmural LV Apex,control,10x5',NA,NA,NA,NaN,NA,nucleus
AAACCTGCAGACAGGT-1--GSM5572792,2318.0,1327,Pericyte,GSM5572792,GSE183852,control,heart,Transmural LV Apex,control,10x5',NA,NA,NA,NaN,NA,nucleus
AAACCTGCAGATCTGT-1--GSM5572792,2977.0,1691,Venous EC,GSM5572792,GSE183852,control,heart,Transmural LV Apex,control,10x5',NA,NA,NA,NaN,NA,nucleus
AAACCTGCAGGATCGA-1--GSM5572792,3496.0,2122,Monocyte,GSM5572792,GSE183852,control,heart,Transmural LV Apex,control,10x5',NA,NA,NA,NaN,NA,nucleus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTCAGTTCGTGGACC-1--GSM5415725,7255.0,2370,Dendritic cell,GSM5415725,GSE179343,control,heart,NA,control,10x3',NA,immune cell,NA,46.0,NA,cell
TTTCATGGTATCGCGC-1--GSM5415725,5435.0,2001,Dendritic cell,GSM5415725,GSE179343,control,heart,NA,control,10x3',NA,immune cell,NA,46.0,NA,cell
TTTCCTCCATCAGTGT-1--GSM5415725,5824.0,2148,Dendritic cell,GSM5415725,GSE179343,control,heart,NA,control,10x3',NA,immune cell,NA,46.0,NA,cell
TTTCGATGTAGGAAAG-1--GSM5415725,6531.0,2140,Dendritic cell,GSM5415725,GSE179343,control,heart,NA,control,10x3',NA,immune cell,NA,46.0,NA,cell


In [66]:
adata.var

,gene
MIR1302-2HG,MIR1302-2HG
FAM138A,FAM138A
OR4F5,OR4F5
AL627309.1,AL627309.1
AL627309.3,AL627309.3
...,...
AC233755.2,AC233755.2
AC233755.1,AC233755.1
AC240274.1,AC240274.1
AC213203.1,AC213203.1


In [67]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [68]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 624358 × 33234
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST'
    var: 'gene_name'


In [69]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
Ventricular cardiomyocyte              191284
CFD+MGP+ fibroblast                    109830
Pericyte                                87675
Capillary EC                            59174
LYVE1 macrophage                        35530
Vascular smooth muscle cell             20580
Arterial EC                             16654
Atrial cardiomyocyte                    15804
Venous EC                               15584
Endocardial cell                        14711
Fetal ventricular cardiomyocyte         14558
Neuron                                   6361
T cell                                   6239
Dendritic cell                           5647
ADAM12+ fibroblast                       4151
Mast cell                                3267
NK cell                                  3026
Adipocyte                                2810
Monocyte                                 2661
Cycling LYVE1 macrophage                 2232
Fetal KCNQ1OT1+MIR503HG+ fibroblast      2226
Lymphatic EC   

In [70]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK
    # -------------------------
    "T cell": "T_NK",
    "NK cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "B cell": "B_Plasma",
    "Plasma cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "LYVE1 macrophage": "Myeloid",
    "Cycling LYVE1 macrophage": "Myeloid",
    "Monocyte": "Myeloid",
    "Dendritic cell": "Myeloid",
    "Mast cell": "Myeloid",

    # -------------------------
    # Blood vessel
    # -------------------------
    "Pericyte": "Blood_vessel",
    "Capillary EC": "Blood_vessel",
    "Arterial EC": "Blood_vessel",
    "Venous EC": "Blood_vessel",
    "Lymphatic EC": "Blood_vessel",
    "Endocardial cell": "Blood_vessel",
    "Vascular smooth muscle cell": "Blood_vessel",

    # -------------------------
    # Fibroblast / Myofibroblast
    # -------------------------
    "CFD+MGP+ fibroblast": "Fibroblast_Myofibroblast",
    "ADAM12+ fibroblast": "Fibroblast_Myofibroblast",
    "Fetal KCNQ1OT1+MIR503HG+ fibroblast": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "Mesothelial cell": "Epithelial",

    # -------------------------
    # Specialized
    # -------------------------
    "Ventricular cardiomyocyte": "Specialized",
    "Atrial cardiomyocyte": "Specialized",
    "Fetal ventricular cardiomyocyte": "Specialized",
    "Neuron": "Specialized",
    "Adipocyte": "Specialized",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
Specialized                 230817
Blood_vessel                216529
Fibroblast_Myofibroblast    116207
Myeloid                      49337
T_NK                          9265
Epithelial                    1331
B_Plasma                       872
Melanocyte                       0
Unknown                          0
Name: count, dtype: int64


In [71]:
new_adata

AnnData object with n_obs × n_vars = 624358 × 33234
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [72]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [73]:
report

{'n_input_genes': 33234,
 'n_mapped_genes': 23425,
 'pct_genes_mapped': 70.48504543539748,
 'n_output_genes_after_collapse': 23353,
 'n_ambiguous_genes': 16,
 'n_unmapped_or_no_ensembl': 9793,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 9809,
 'n_duplicate_ensembl_ids': 68,
 'n_duplicate_input_features_collapsed': 72,
 'total_counts_before': 1831474393.0,
 'total_counts_retained': 1786593519.0,
 'pct_total_counts_retained': 97.5494675671395,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [74]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
MIR1302-2HG,MIR1302-2HG,MIR1302-2HG,ENSG00000243485,approved_symbol
FAM138A,FAM138A,FAM138A,ENSG00000237613,approved_symbol
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
...,...,...,...,...
AC233755.2,AC233755.2,<NA>,<NA>,unmapped_or_no_ensembl
AC233755.1,AC233755.1,<NA>,<NA>,unmapped_or_no_ensembl
AC240274.1,AC240274.1,<NA>,<NA>,unmapped_or_no_ensembl
AC213203.1,AC213203.1,<NA>,<NA>,unmapped_or_no_ensembl


In [75]:
adata_ensembl

AnnData object with n_obs × n_vars = 624358 × 23353
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [76]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Kidney

In [5]:
tissue = 'Kidney'

In [6]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

$HOME/miniforge3/envs/process_slide/lib/python3.12/site-packages/anndata/_io/h5ad.py:266: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  return AnnData(**{


AnnData object with n_obs × n_vars = 103547 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'disease_subtype', 'time_point', 'subject_id', 'age', 'gender', 'race', 'disease_stage', 'rna_source', 'other_metadata'
    var: 'gene'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances'

In [7]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,anatomical_site,disease,platform,cell_sorting,disease_subtype,time_point,subject_id,age,gender,race,disease_stage,rna_source,other_metadata
AAACCTGAGCGATATA-1--GSM4339775,1521.0,844,Capillary EC,GSM4339775,GSE145927,allograft,kidney,NA,non-small cell lung cancer,10x5',NA,NA,28 days,NA,NaN,NA,NA,NA,cell,NA
AAACCTGAGCTAGTCT-1--GSM4339775,1761.0,894,Capillary EC,GSM4339775,GSE145927,allograft,kidney,NA,non-small cell lung cancer,10x5',NA,NA,28 days,NA,NaN,NA,NA,NA,cell,NA
AAACCTGAGGTACTCT-1--GSM4339775,2317.0,1189,Arterial EC,GSM4339775,GSE145927,allograft,kidney,NA,non-small cell lung cancer,10x5',NA,NA,28 days,NA,NaN,NA,NA,NA,cell,NA
AAACCTGAGTTAAGTG-1--GSM4339775,2311.0,1052,Capillary EC,GSM4339775,GSE145927,allograft,kidney,NA,non-small cell lung cancer,10x5',NA,NA,28 days,NA,NaN,NA,NA,NA,cell,NA
AAACCTGCAAAGCGGT-1--GSM4339775,5571.0,2163,Renal epithelial cell (Loop of Henle),GSM4339775,GSE145927,allograft,kidney,NA,non-small cell lung cancer,10x5',NA,NA,28 days,NA,NaN,NA,NA,NA,cell,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCAAGATATGCA-1--GSM4145204,1264.0,603,Renal epithelial cell (proximal tubule),GSM4145204,GSE131685,adjacent normal,kidney,NA,papillary urothelial carcinoma,10x3',NA,papillary urothelial carcinoma of the renal pe...,NA,NA,57.0,M,NA,NA,cell,{BMI: 25.6}
TTTGTCAAGTGGGATC-1--GSM4145204,2155.0,856,Renal epithelial cell (proximal tubule),GSM4145204,GSE131685,adjacent normal,kidney,NA,papillary urothelial carcinoma,10x3',NA,papillary urothelial carcinoma of the renal pe...,NA,NA,57.0,M,NA,NA,cell,{BMI: 25.6}
TTTGTCACATGCGCAC-1--GSM4145204,1301.0,725,Renal epithelial cell (proximal tubule),GSM4145204,GSE131685,adjacent normal,kidney,NA,papillary urothelial carcinoma,10x3',NA,papillary urothelial carcinoma of the renal pe...,NA,NA,57.0,M,NA,NA,cell,{BMI: 25.6}
TTTGTCAGTTTGACAC-1--GSM4145204,3197.0,1246,Renal epithelial cell (proximal tubule),GSM4145204,GSE131685,adjacent normal,kidney,NA,papillary urothelial carcinoma,10x3',NA,papillary urothelial carcinoma of the renal pe...,NA,NA,57.0,M,NA,NA,cell,{BMI: 25.6}


In [8]:
adata.var

,gene
MIR1302-2HG,MIR1302-2HG
FAM138A,FAM138A
OR4F5,OR4F5
AL627309.1,AL627309.1
AL627309.3,AL627309.3
...,...
AC233755.2,AC233755.2
AC233755.1,AC233755.1
AC240274.1,AC240274.1
AC213203.1,AC213203.1


In [9]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [10]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 103547 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'disease_subtype', 'time_point', 'subject_id', 'age', 'gender', 'race', 'disease_stage', 'rna_source', 'other_metadata', 'tissue_HistoST'
    var: 'gene_name'


In [11]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
Renal epithelial cell (proximal tubule)    31913
Renal epithelial cell (distal tubule)      11560
Renal epithelial cell (Loop of Henle)      11133
Venous EC                                   8468
CD14 monocyte                               5998
CD4 T cell                                  4081
Capillary EC                                3338
Type A intercalated cell                    3257
NK cell                                     3126
CD8 T cell                                  3072
Macrophage                                  2858
CD16 monocyte                               2498
Arterial EC                                 2387
Fibroblast                                  2290
cDC2                                        1622
Type B intercalated cell                     988
Plasma cell                                  974
Podocyte                                     823
Cycling T/NK cell                            804
Treg cell                                    636
P

In [12]:
# ct_1 mapping

# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK
    # -------------------------
    "CD4 T cell": "T_NK",
    "CD8 T cell": "T_NK",
    "NK cell": "T_NK",
    "Cycling T/NK cell": "T_NK",
    "Treg cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "B cell": "B_Plasma",
    "Plasma cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "CD14 monocyte": "Myeloid",
    "CD16 monocyte": "Myeloid",
    "Macrophage": "Myeloid",
    "cDC1": "Myeloid",
    "cDC2": "Myeloid",
    "Cycling myeloid cell": "Myeloid",

    # -------------------------
    # Blood vessel
    # -------------------------
    "Arterial EC": "Blood_vessel",
    "Capillary EC": "Blood_vessel",
    "Venous EC": "Blood_vessel",
    "Lymphatic EC": "Blood_vessel",
    "Pericyte": "Blood_vessel",

    # -------------------------
    # Fibroblast / Myofibroblast
    # -------------------------
    "Fibroblast": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "Renal epithelial cell (proximal tubule)": "Epithelial",
    "Renal epithelial cell (distal tubule)": "Epithelial",
    "Renal epithelial cell (Loop of Henle)": "Epithelial",
    "Type A intercalated cell": "Epithelial",
    "Type B intercalated cell": "Epithelial",

    # -------------------------
    # Specialized
    # -------------------------
    "Podocyte": "Specialized",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
Epithelial                  58851
Blood_vessel                14928
Myeloid                     13466
T_NK                        11719
Fibroblast_Myofibroblast     2290
B_Plasma                     1470
Specialized                   823
Melanocyte                      0
Unknown                         0
Name: count, dtype: int64


In [13]:
new_adata

AnnData object with n_obs × n_vars = 103547 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'disease_subtype', 'time_point', 'subject_id', 'age', 'gender', 'race', 'disease_stage', 'rna_source', 'other_metadata', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [14]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [15]:
report

{'n_input_genes': 33538,
 'n_mapped_genes': 23480,
 'pct_genes_mapped': 70.01013775418927,
 'n_output_genes_after_collapse': 23398,
 'n_ambiguous_genes': 16,
 'n_unmapped_or_no_ensembl': 10042,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 10058,
 'n_duplicate_ensembl_ids': 77,
 'n_duplicate_input_features_collapsed': 82,
 'total_counts_before': 320896286.0,
 'total_counts_retained': 318856846.0,
 'pct_total_counts_retained': 99.36445509375574,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [16]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
MIR1302-2HG,MIR1302-2HG,MIR1302-2HG,ENSG00000243485,approved_symbol
FAM138A,FAM138A,FAM138A,ENSG00000237613,approved_symbol
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
...,...,...,...,...
AC233755.2,AC233755.2,<NA>,<NA>,unmapped_or_no_ensembl
AC233755.1,AC233755.1,<NA>,<NA>,unmapped_or_no_ensembl
AC240274.1,AC240274.1,<NA>,<NA>,unmapped_or_no_ensembl
AC213203.1,AC213203.1,<NA>,<NA>,unmapped_or_no_ensembl


In [17]:
adata_ensembl

AnnData object with n_obs × n_vars = 103547 × 23398
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'disease_subtype', 'time_point', 'subject_id', 'age', 'gender', 'race', 'disease_stage', 'rna_source', 'other_metadata', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [18]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Liver

In [19]:
tissue = 'Liver'

In [20]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

$HOME/miniforge3/envs/process_slide/lib/python3.12/site-packages/anndata/_io/h5ad.py:266: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  return AnnData(**{


AnnData object with n_obs × n_vars = 367596 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'disease_subtype', 'treatment', 'time_point', 'subject_id', 'age', 'gender', 'race', 'infection', 'disease_stage', 'rna_source', 'other_metadata'
    var: 'gene'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_umap'
    obsp: 'distances'

In [21]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,anatomical_site,disease,platform,...,treatment,time_point,subject_id,age,gender,race,infection,disease_stage,rna_source,other_metadata
CZI-IA10466281+CZI-IA10466377_AAAGCAACAACAACCT--637C-LIV,5191.0,2330,Memory CD4 T cell,637C-LIV,E-MTAB-11536,control,liver,NA,control,10x5',...,NA,NA,637C,NaN,M,NA,NA,NA,cell,NA
CZI-IA10466281+CZI-IA10466377_AAAGTAGGTGTGACCC--637C-LIV,6819.0,2608,Memory CD4 T cell,637C-LIV,E-MTAB-11536,control,liver,NA,control,10x5',...,NA,NA,637C,NaN,M,NA,NA,NA,cell,NA
CZI-IA10466281+CZI-IA10466377_AACACGTCAGCATACT--637C-LIV,5198.0,1795,Memory CD4 T cell,637C-LIV,E-MTAB-11536,control,liver,NA,control,10x5',...,NA,NA,637C,NaN,M,NA,NA,NA,cell,NA
CZI-IA10466281+CZI-IA10466377_AACTCAGTCCTATGTT--637C-LIV,8994.0,2298,CD14 monocyte,637C-LIV,E-MTAB-11536,control,liver,NA,control,10x5',...,NA,NA,637C,NaN,M,NA,NA,NA,cell,NA
CZI-IA10466281+CZI-IA10466377_AACTCCCAGAGCTGGT--637C-LIV,7395.0,2385,CD14 monocyte,637C-LIV,E-MTAB-11536,control,liver,NA,control,10x5',...,NA,NA,637C,NaN,M,NA,NA,NA,cell,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCAGTTCTGGTA-1--SRX19743975,997.0,678,CD16 NK cell,SRX19743975,PRJNA947270,adjacent normal,liver,NA,hepatocellular carcinoma,10x5',...,NA,NA,P3,61.0,F,NA,HBV+HCV-,NA,cell,NA
TTTGTCATCCCGACTT-1--SRX19743975,3310.0,1141,Treg cell,SRX19743975,PRJNA947270,adjacent normal,liver,NA,hepatocellular carcinoma,10x5',...,NA,NA,P3,61.0,F,NA,HBV+HCV-,NA,cell,NA
TTTGTCATCCGAAGAG-1--SRX19743975,2707.0,1461,CXCL13 exhausted CD8 T cell,SRX19743975,PRJNA947270,adjacent normal,liver,NA,hepatocellular carcinoma,10x5',...,NA,NA,P3,61.0,F,NA,HBV+HCV-,NA,cell,NA
TTTGTCATCGGACAAG-1--SRX19743975,1829.0,1039,CXCL13 exhausted CD8 T cell,SRX19743975,PRJNA947270,adjacent normal,liver,NA,hepatocellular carcinoma,10x5',...,NA,NA,P3,61.0,F,NA,HBV+HCV-,NA,cell,NA


In [22]:
adata.var

,gene
MIR1302-2HG,MIR1302-2HG
FAM138A,FAM138A
OR4F5,OR4F5
AL627309.1,AL627309.1
AL627309.3,AL627309.3
...,...
C21orf2,C21orf2
AP001065.1,AP001065.1
AP001065.2,AP001065.2
AC004556.1,AC004556.1


In [23]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [24]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 367596 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'disease_subtype', 'treatment', 'time_point', 'subject_id', 'age', 'gender', 'race', 'infection', 'disease_stage', 'rna_source', 'other_metadata', 'tissue_HistoST'
    var: 'gene_name'


In [25]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
GZMK CD8 T cell                    45348
CD14 monocyte                      44291
Memory CD4 T cell                  37267
CD16 NK cell                       24538
MAIT cell                          18517
Hepatocyte                         13260
CD16 monocyte                      12752
CD56 NK cell                       12678
Kupffer cell                       12539
Naive CD4 T cell                   10325
Macrophage                         10110
cDC2                               10092
GZMB CD8 T cell                     9903
Plasma cell                         9078
Memory B cell                       7791
LSEC                                7630
Cycling T/NK cell                   7345
Naive CD8 T cell                    7047
Neutrophil                          6830
Naive B cell                        5983
Arterial EC                         5230
XCL1 NK cell                        4912
Small cholangiocyte                 4892
Portal venous EC                    41

In [26]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK / innate lymphoid
    # -------------------------
    "GZMK CD8 T cell": "T_NK",
    "Memory CD4 T cell": "T_NK",
    "CD16 NK cell": "T_NK",
    "MAIT cell": "T_NK",
    "CD56 NK cell": "T_NK",
    "Naive CD4 T cell": "T_NK",
    "GZMB CD8 T cell": "T_NK",
    "Cycling T/NK cell": "T_NK",
    "Naive CD8 T cell": "T_NK",
    "XCL1 NK cell": "T_NK",
    "CXCL13 exhausted CD8 T cell": "T_NK",
    "Gamma delta T cell": "T_NK",
    "Treg cell": "T_NK",
    "ILC": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "Plasma cell": "B_Plasma",
    "Memory B cell": "B_Plasma",
    "Naive B cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "CD14 monocyte": "Myeloid",
    "CD16 monocyte": "Myeloid",
    "Kupffer cell": "Myeloid",
    "Macrophage": "Myeloid",
    "cDC2": "Myeloid",
    "Neutrophil": "Myeloid",
    "cDC1": "Myeloid",
    "pDC": "Myeloid",
    "Mast cell": "Myeloid",
    "Metamyelocyte/Band neutrophil": "Myeloid",
    "SPP1 macrophage": "Myeloid",

    # -------------------------
    # Blood vessel / endothelial
    # -------------------------
    "LSEC": "Blood_vessel",
    "Arterial EC": "Blood_vessel",
    "Portal venous EC": "Blood_vessel",
    "Capillary EC": "Blood_vessel",
    "Central venous EC": "Blood_vessel",
    "Venous EC": "Blood_vessel",
    "Lymphatic EC": "Blood_vessel",
    "EndoMT": "Blood_vessel",

    # -------------------------
    # Fibroblast / myofibroblast / stromal
    # -------------------------
    "Quiescent hepatic stellate cell": "Fibroblast_Myofibroblast",
    "Activated hepatic stellate cell": "Fibroblast_Myofibroblast",
    "Mesothelial cell": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "Hepatocyte": "Epithelial",
    "Small cholangiocyte": "Epithelial",
    "Large cholangiocyte": "Epithelial",

    # -------------------------
    # Specialized
    # -------------------------
    "Erythroblast": "Specialized",
    "Neuron": "Specialized",
    "Glial cell": "Specialized",

    # -------------------------
    # Optional explicit fallback labels
    # -------------------------
    "Unknown": "Unknown",
    "ToRemove": "Unknown",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
T_NK                        188639
Myeloid                     105336
Blood_vessel                 23033
B_Plasma                     22852
Epithelial                   19471
Specialized                   4788
Fibroblast_Myofibroblast      3477
Melanocyte                       0
Unknown                          0
Name: count, dtype: int64


In [27]:
new_adata

AnnData object with n_obs × n_vars = 367596 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'disease_subtype', 'treatment', 'time_point', 'subject_id', 'age', 'gender', 'race', 'infection', 'disease_stage', 'rna_source', 'other_metadata', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [28]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [29]:
report

{'n_input_genes': 33538,
 'n_mapped_genes': 23480,
 'pct_genes_mapped': 70.01013775418927,
 'n_output_genes_after_collapse': 23398,
 'n_ambiguous_genes': 16,
 'n_unmapped_or_no_ensembl': 10042,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 10058,
 'n_duplicate_ensembl_ids': 77,
 'n_duplicate_input_features_collapsed': 82,
 'total_counts_before': 1868820262.0,
 'total_counts_retained': 1849994803.0,
 'pct_total_counts_retained': 98.99265545313314,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [30]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
MIR1302-2HG,MIR1302-2HG,MIR1302-2HG,ENSG00000243485,approved_symbol
FAM138A,FAM138A,FAM138A,ENSG00000237613,approved_symbol
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
...,...,...,...,...
C21orf2,C21orf2,CFAP410,ENSG00000160226,alias_or_previous_symbol
AP001065.1,AP001065.1,<NA>,<NA>,unmapped_or_no_ensembl
AP001065.2,AP001065.2,<NA>,<NA>,unmapped_or_no_ensembl
AC004556.1,AC004556.1,<NA>,<NA>,unmapped_or_no_ensembl


In [31]:
adata_ensembl

AnnData object with n_obs × n_vars = 367596 × 23398
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'disease_subtype', 'treatment', 'time_point', 'subject_id', 'age', 'gender', 'race', 'infection', 'disease_stage', 'rna_source', 'other_metadata', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [32]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Lung

In [33]:
tissue = 'Lung'

In [34]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 221012 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'disease_subtype', 'treatment', 'subject_id', 'age', 'gender', 'race', 'infection', 'disease_stage', 'rna_source', 'other_metadata'
    var: 'gene'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'

In [35]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,anatomical_site,disease,platform,...,disease_subtype,treatment,subject_id,age,gender,race,infection,disease_stage,rna_source,other_metadata
AAACCTGAGACAGGCT-1--ERX2757114,4748.0,1543,M1 macrophage,ERX2757114,E-MTAB-6653,adjacent normal,lung,normal tissue adjacent to tumour,lung carcinoma,10x3',...,NA,NA,8,55.0,F,NA,NA,NA,cell,NA
AAACCTGCAAGGGTCA-1--ERX2757114,5917.0,1644,Alveolar macrophage,ERX2757114,E-MTAB-6653,adjacent normal,lung,normal tissue adjacent to tumour,lung carcinoma,10x3',...,NA,NA,8,55.0,F,NA,NA,NA,cell,NA
AAACCTGCACAGCGTC-1--ERX2757114,2885.0,1031,CD16 monocyte,ERX2757114,E-MTAB-6653,adjacent normal,lung,normal tissue adjacent to tumour,lung carcinoma,10x3',...,NA,NA,8,55.0,F,NA,NA,NA,cell,NA
AAACCTGCACGCTTTC-1--ERX2757114,12407.0,2421,Alveolar macrophage,ERX2757114,E-MTAB-6653,adjacent normal,lung,normal tissue adjacent to tumour,lung carcinoma,10x3',...,NA,NA,8,55.0,F,NA,NA,NA,cell,NA
AAACCTGCAGACGTAG-1--ERX2757114,2762.0,1153,Naive B cell,ERX2757114,E-MTAB-6653,adjacent normal,lung,normal tissue adjacent to tumour,lung carcinoma,10x3',...,NA,NA,8,55.0,F,NA,NA,NA,cell,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTGCATTTCCTAACAG-1--GSM5020384,13870.0,3790,Alveolar macrophage,GSM5020384,GSE164829,control,lung,NA,control,10x3',...,NA,NA,NA,NaN,NA,NA,NA,NA,cell,NA
TTGTTGTCATCTTTCA-1--GSM5020384,21899.0,5278,ADAMDEC1+ADAM28+ fibroblast,GSM5020384,GSE164829,control,lung,NA,control,10x3',...,NA,NA,NA,NaN,NA,NA,NA,NA,cell,NA
TTTACCACAAATGAAC-1--GSM5020384,2495.0,1183,Alveolar macrophage,GSM5020384,GSE164829,control,lung,NA,control,10x3',...,NA,NA,NA,NaN,NA,NA,NA,NA,cell,NA
TTTATGCAGGCTCCCA-1--GSM5020384,17324.0,4744,ADAMDEC1+ADAM28+ fibroblast,GSM5020384,GSE164829,control,lung,NA,control,10x3',...,NA,NA,NA,NaN,NA,NA,NA,NA,cell,NA


In [36]:
adata.var

,gene
MIR1302-2HG,MIR1302-2HG
FAM138A,FAM138A
OR4F5,OR4F5
AL627309.1,AL627309.1
AL627309.3,AL627309.3
...,...
AC233755.2,AC233755.2
AC233755.1,AC233755.1
AC240274.1,AC240274.1
AC213203.1,AC213203.1


In [37]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [38]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 221012 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'disease_subtype', 'treatment', 'subject_id', 'age', 'gender', 'race', 'infection', 'disease_stage', 'rna_source', 'other_metadata', 'tissue_HistoST'
    var: 'gene_name'


In [39]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
Alveolar macrophage            45378
Memory CD4 T cell              18313
CFD+MGP+ fibroblast            16312
AT2                            13653
Naive CD4 T cell               13116
M1 macrophage                  12645
GZMB CD8 T cell                10498
CD16 NK cell                    9055
CD14 monocyte                   8005
Treg cell                       7311
Capillary EC                    7154
GZMK CD8 T cell                 5707
Aerocyte                        4948
Vascular smooth muscle cell     4515
AT1                             3645
Multiciliated cell              3235
cDC2                            3151
Myofibroblast                   3083
Mast cell                       2847
LYVE1 macrophage                2722
Venous EC                       2690
Pericyte                        2395
Arterial EC                     2201
CD16 monocyte                   2191
Club cell                       2106
Airway basal cell               2020
GPC3+ fibroblast   

In [40]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK
    # -------------------------
    "Memory CD4 T cell": "T_NK",
    "Naive CD4 T cell": "T_NK",
    "GZMB CD8 T cell": "T_NK",
    "GZMK CD8 T cell": "T_NK",
    "CD16 NK cell": "T_NK",
    "Treg cell": "T_NK",
    "Cycling T/NK cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "Memory B cell": "B_Plasma",
    "Naive B cell": "B_Plasma",
    "Plasma cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "Alveolar macrophage": "Myeloid",
    "M1 macrophage": "Myeloid",
    "LYVE1 macrophage": "Myeloid",
    "CD14 monocyte": "Myeloid",
    "CD16 monocyte": "Myeloid",
    "cDC1": "Myeloid",
    "cDC2": "Myeloid",
    "mregDC": "Myeloid",
    "Mast cell": "Myeloid",
    "G2/M phase myeloid cell": "Myeloid",
    "S phase myeloid cell": "Myeloid",

    # -------------------------
    # Blood vessel
    # -------------------------
    "Capillary EC": "Blood_vessel",
    "Aerocyte": "Blood_vessel",
    "Venous EC": "Blood_vessel",
    "Arterial EC": "Blood_vessel",
    "Lymphatic EC": "Blood_vessel",
    "Pericyte": "Blood_vessel",
    "Vascular smooth muscle cell": "Blood_vessel",

    # -------------------------
    # Fibroblast / Myofibroblast
    # -------------------------
    "CFD+MGP+ fibroblast": "Fibroblast_Myofibroblast",
    "GPC3+ fibroblast": "Fibroblast_Myofibroblast",
    "ADAMDEC1+ADAM28+ fibroblast": "Fibroblast_Myofibroblast",
    "Myofibroblast": "Fibroblast_Myofibroblast",
    "S phase GPC3+ fibroblast": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "AT1": "Epithelial",
    "AT2": "Epithelial",
    "Cycling AT2": "Epithelial",
    "Club cell": "Epithelial",
    "Airway basal cell": "Epithelial",
    "Goblet cell": "Epithelial",
    "Multiciliated cell": "Epithelial",
    "CDC20B+ Multiciliated cell": "Epithelial",
    "Neuroendocrine cell": "Epithelial",

    # -------------------------
    # Specialized
    # -------------------------
    "Red blood cell": "Specialized",
    "Neuron": "Specialized",

    # -------------------------
    # Optional explicit unknown
    # -------------------------
    "Unknown": "Unknown",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
Myeloid                     78447
T_NK                        65119
Epithelial                  26540
Blood_vessel                25126
Fibroblast_Myofibroblast    21877
B_Plasma                     3160
Specialized                   743
Melanocyte                      0
Unknown                         0
Name: count, dtype: int64


In [41]:
new_adata

AnnData object with n_obs × n_vars = 221012 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'disease_subtype', 'treatment', 'subject_id', 'age', 'gender', 'race', 'infection', 'disease_stage', 'rna_source', 'other_metadata', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [42]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [43]:
report

{'n_input_genes': 33538,
 'n_mapped_genes': 23480,
 'pct_genes_mapped': 70.01013775418927,
 'n_output_genes_after_collapse': 23398,
 'n_ambiguous_genes': 16,
 'n_unmapped_or_no_ensembl': 10042,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 10058,
 'n_duplicate_ensembl_ids': 77,
 'n_duplicate_input_features_collapsed': 82,
 'total_counts_before': 1105279466.0,
 'total_counts_retained': 1097962501.0,
 'pct_total_counts_retained': 99.3379986487508,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [44]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
MIR1302-2HG,MIR1302-2HG,MIR1302-2HG,ENSG00000243485,approved_symbol
FAM138A,FAM138A,FAM138A,ENSG00000237613,approved_symbol
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
...,...,...,...,...
AC233755.2,AC233755.2,<NA>,<NA>,unmapped_or_no_ensembl
AC233755.1,AC233755.1,<NA>,<NA>,unmapped_or_no_ensembl
AC240274.1,AC240274.1,<NA>,<NA>,unmapped_or_no_ensembl
AC213203.1,AC213203.1,<NA>,<NA>,unmapped_or_no_ensembl


In [45]:
adata_ensembl

AnnData object with n_obs × n_vars = 221012 × 23398
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'disease_subtype', 'treatment', 'subject_id', 'age', 'gender', 'race', 'infection', 'disease_stage', 'rna_source', 'other_metadata', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [46]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## LymphNode

In [47]:
tissue = 'LymphNode'

In [48]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 273755 × 32577
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source'
    var: 'gene'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'

In [49]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,disease,platform,cell_sorting,subject_id,age,gender,rna_source
wf4su8ny_h4yj8bv7_AAACGAATCCATTGGA-1--BCLL_13_T_1_hashing,1398.0,694,Macrophage,BCLL_13_T_1_hashing,EGAS00001006375,control,tonsil,control,10x3',NA,BCLL-13-T,5.0,F,cell
wf4su8ny_h4yj8bv7_ACGGTTAGTACGTACT-1--BCLL_13_T_1_hashing,1305.0,668,Macrophage,BCLL_13_T_1_hashing,EGAS00001006375,control,tonsil,control,10x3',NA,BCLL-13-T,5.0,F,cell
wf4su8ny_h4yj8bv7_ACTACGAGTTTCACTT-1--BCLL_13_T_1_hashing,1705.0,757,Macrophage,BCLL_13_T_1_hashing,EGAS00001006375,control,tonsil,control,10x3',NA,BCLL-13-T,5.0,F,cell
wf4su8ny_h4yj8bv7_AGAAGCGAGTGCCAGA-1--BCLL_13_T_1_hashing,2634.0,1125,Macrophage,BCLL_13_T_1_hashing,EGAS00001006375,control,tonsil,control,10x3',NA,BCLL-13-T,5.0,F,cell
wf4su8ny_h4yj8bv7_AGCCACGGTGGTAATA-1--BCLL_13_T_1_hashing,3462.0,782,mregDC,BCLL_13_T_1_hashing,EGAS00001006375,control,tonsil,control,10x3',NA,BCLL-13-T,5.0,F,cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCAAGATAGTCA-1--GSM3375767,8965.0,2475,MHCII high CD14 monocyte,GSM3375767,GSE119506,control,tonsil,control,10x3',CD14+ cells,NA,NaN,NA,cell
TTTGTCAAGTTCCACA-1--GSM3375767,7114.0,2007,MHCII high CD14 monocyte,GSM3375767,GSE119506,control,tonsil,control,10x3',CD14+ cells,NA,NaN,NA,cell
TTTGTCACAGTAACGG-1--GSM3375767,5114.0,1700,MHCII high CD14 monocyte,GSM3375767,GSE119506,control,tonsil,control,10x3',CD14+ cells,NA,NaN,NA,cell
TTTGTCAGTCTAACGT-1--GSM3375767,4661.0,1645,MHCII high CD14 monocyte,GSM3375767,GSE119506,control,tonsil,control,10x3',CD14+ cells,NA,NaN,NA,cell


In [50]:
adata.var

,gene
AL627309.1,AL627309.1
AL627309.3,AL627309.3
AL627309.4,AL627309.4
AL669831.2,AL669831.2
FAM87B,FAM87B
...,...
AC171558.1,AC171558.1
AC133551.1,AC133551.1
AC136612.1,AC136612.1
AC136616.1,AC136616.1


In [51]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [52]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 273755 × 32577
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST'
    var: 'gene_name'


In [53]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
Naive B cell                         44781
DZ/LZ B cell                         22913
Unswitched memory B cell             17194
Tfh cell                             16723
Class-switched memory B cell         15635
Memory CD4 T cell                    14767
Non-cycling GC B cell                14341
Naive CD4 T cell                     12791
Light zone GC B cell                 12640
FCRL4 memory B cell                  12567
Committed GC B cell                  10402
Treg cell                            10211
Histone high S phase DZ GC B cell     9343
S phase DZ GC B cell                  9288
Plasma cell                           8117
PIF1+ G2/M phase DZ GC B cell         7274
Memory CD8 T cell                     6046
Memory B cell precursor               4538
Pre-GC B cell                         4316
Naive CD8 T cell                      4213
G2/M phase DZ GC B cell               4049
MHCII high CD14 monocyte              2632
Plasma cell precursor              

In [54]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK
    # -------------------------
    "Tfh cell": "T_NK",
    "Memory CD4 T cell": "T_NK",
    "Naive CD4 T cell": "T_NK",
    "Treg cell": "T_NK",
    "Memory CD8 T cell": "T_NK",
    "Naive CD8 T cell": "T_NK",
    "NK cell": "T_NK",
    "Double negative T cell": "T_NK",
    "Cycling T cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "Naive B cell": "B_Plasma",
    "DZ/LZ B cell": "B_Plasma",
    "Unswitched memory B cell": "B_Plasma",
    "Class-switched memory B cell": "B_Plasma",
    "Non-cycling GC B cell": "B_Plasma",
    "Light zone GC B cell": "B_Plasma",
    "FCRL4 memory B cell": "B_Plasma",
    "Committed GC B cell": "B_Plasma",
    "Histone high S phase DZ GC B cell": "B_Plasma",
    "S phase DZ GC B cell": "B_Plasma",
    "Plasma cell": "B_Plasma",
    "PIF1+ G2/M phase DZ GC B cell": "B_Plasma",
    "Memory B cell precursor": "B_Plasma",
    "Pre-GC B cell": "B_Plasma",
    "G2/M phase DZ GC B cell": "B_Plasma",
    "Plasma cell precursor": "B_Plasma",
    "INF-activated naive B cell": "B_Plasma",
    "PreB cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "MHCII high CD14 monocyte": "Myeloid",
    "Macrophage": "Myeloid",
    "pDC": "Myeloid",
    "cDC2": "Myeloid",
    "mregDC": "Myeloid",
    "Granulocyte": "Myeloid",
    "Light zone follicular DC": "Myeloid",
    "Dark zone follicular DC": "Myeloid",

    # -------------------------
    # Epithelial
    # -------------------------
    "Tonsillar crypt epithelial cell": "Epithelial",
    "Epithelial cell": "Epithelial",

    # -------------------------
    # Optional robustness
    # -------------------------
    "Unknown": "Unknown",
    "ToRemove": "Unknown",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
B_Plasma                    201142
T_NK                         66581
Myeloid                       5785
Epithelial                     247
Blood_vessel                     0
Fibroblast_Myofibroblast         0
Specialized                      0
Melanocyte                       0
Unknown                          0
Name: count, dtype: int64


In [55]:
new_adata

AnnData object with n_obs × n_vars = 273755 × 32577
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [56]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [57]:
report

{'n_input_genes': 32577,
 'n_mapped_genes': 23361,
 'pct_genes_mapped': 71.71010221935721,
 'n_output_genes_after_collapse': 23284,
 'n_ambiguous_genes': 15,
 'n_unmapped_or_no_ensembl': 9201,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 9216,
 'n_duplicate_ensembl_ids': 73,
 'n_duplicate_input_features_collapsed': 77,
 'total_counts_before': 1498933174.0,
 'total_counts_retained': 1490047146.0,
 'pct_total_counts_retained': 99.4071765069895,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [58]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.4,AL627309.4,<NA>,<NA>,unmapped_or_no_ensembl
AL669831.2,AL669831.2,<NA>,<NA>,unmapped_or_no_ensembl
FAM87B,FAM87B,FAM87B,ENSG00000177757,approved_symbol
...,...,...,...,...
AC171558.1,AC171558.1,<NA>,<NA>,unmapped_or_no_ensembl
AC133551.1,AC133551.1,<NA>,<NA>,unmapped_or_no_ensembl
AC136612.1,AC136612.1,<NA>,<NA>,unmapped_or_no_ensembl
AC136616.1,AC136616.1,<NA>,<NA>,unmapped_or_no_ensembl


In [59]:
adata_ensembl

AnnData object with n_obs × n_vars = 273755 × 23284
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [60]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Ovary

In [5]:
tissue = 'Ovary'

In [6]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 124128 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'age_group', 'subject_id', 'age', 'gender', 'rna_source'
    var: 'gene'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'

In [7]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,disease,platform,age_group,subject_id,age,gender,rna_source
AAACCTGAGACCACGA-1--ERX9230806,6741.0,2754,GPC3+ fibroblast,ERX9230806,E-MTAB-10551,control,ovary,control,10x5',12 gestational week,Hrv30,NaN,F,cell
AAACCTGAGAGGGATA-1--ERX9230806,4980.0,2314,MT2A+ granulosa cell,ERX9230806,E-MTAB-10551,control,ovary,control,10x5',12 gestational week,Hrv30,NaN,F,cell
AAACCTGAGAGGGCTT-1--ERX9230806,3157.0,1643,OSR1+ granulosa cell,ERX9230806,E-MTAB-10551,control,ovary,control,10x5',12 gestational week,Hrv30,NaN,F,cell
AAACCTGAGAGTCTGG-1--ERX9230806,13661.0,4401,MT2A+ granulosa cell,ERX9230806,E-MTAB-10551,control,ovary,control,10x5',12 gestational week,Hrv30,NaN,F,cell
AAACCTGAGGGCTTCC-1--ERX9230806,7981.0,3228,MT2A+ granulosa cell,ERX9230806,E-MTAB-10551,control,ovary,control,10x5',12 gestational week,Hrv30,NaN,F,cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGCGCCATGGATGG-1--ERX9230814,6900.0,3103,Podocyte,ERX9230814,E-MTAB-10551,control,ovary,control,10x5',6 gestational week,F123,NaN,F,cell
TTTGGTTCAAATTGCC-1--ERX9230814,13208.0,4583,Podocyte,ERX9230814,E-MTAB-10551,control,ovary,control,10x5',6 gestational week,F123,NaN,F,cell
TTTGGTTGTGCCTGCA-1--ERX9230814,10451.0,4075,Podocyte,ERX9230814,E-MTAB-10551,control,ovary,control,10x5',6 gestational week,F123,NaN,F,cell
TTTGTCAAGCGTTTAC-1--ERX9230814,18602.0,5306,CLDN3+ epithelial cell,ERX9230814,E-MTAB-10551,control,ovary,control,10x5',6 gestational week,F123,NaN,F,cell


In [8]:
adata.var

,gene
MIR1302-2HG,MIR1302-2HG
FAM138A,FAM138A
OR4F5,OR4F5
AL627309.1,AL627309.1
AL627309.3,AL627309.3
...,...
AC233755.2,AC233755.2
AC233755.1,AC233755.1
AC240274.1,AC240274.1
AC213203.1,AC213203.1


In [9]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [10]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 124128 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'age_group', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST'
    var: 'gene_name'


In [11]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
GPC3+ fibroblast                          37361
MT2A+ granulosa cell                      24756
BNC2+ZFPM2+ fibroblast                    11504
OSR1+ granulosa cell                       6628
S phase GPC3+ fibroblast                   5832
GATA4+ mesothelial cell                    5029
TAC1+ granulosa cell                       4836
Capillary EC                               2782
PAX8+ granulosa cell                       2302
G2/M phase GPC3+ fibroblast                2042
CRABP1+ mesothelial cell                   1823
CD16 NK cell                               1643
Pericyte                                   1573
CD8 T cell                                 1568
CLDN3+ epithelial cell                     1559
Macrophage                                 1535
Red blood cell                             1116
Cycling mesothelial cell                   1111
Podocyte                                    772
AFP+ epithelial cell                        761
Memory CD4 T cell    

In [12]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK
    # -------------------------
    "CD16 NK cell": "T_NK",
    "CD8 T cell": "T_NK",
    "Memory CD4 T cell": "T_NK",
    "Double negative T cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "B cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "Macrophage": "Myeloid",
    "Cycling macrophage": "Myeloid",
    "Monocyte": "Myeloid",

    # -------------------------
    # Blood vessel
    # -------------------------
    "Capillary EC": "Blood_vessel",
    "Cycling EC": "Blood_vessel",
    "Arterial EC": "Blood_vessel",
    "Lymphatic EC": "Blood_vessel",
    "Venous EC": "Blood_vessel",
    "Pericyte": "Blood_vessel",
    "SPRR2F+ pericyte": "Blood_vessel",
    "Vascular smooth muscle cell": "Blood_vessel",
    "CREB+MT1A+ vascular smooth muscle cell": "Blood_vessel",
    "Cycling perivascular cell": "Blood_vessel",

    # -------------------------
    # Fibroblast / Myofibroblast
    # -------------------------
    "GPC3+ fibroblast": "Fibroblast_Myofibroblast",
    "BNC2+ZFPM2+ fibroblast": "Fibroblast_Myofibroblast",
    "S phase GPC3+ fibroblast": "Fibroblast_Myofibroblast",
    "G2/M phase GPC3+ fibroblast": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "CLDN3+ epithelial cell": "Epithelial",
    "AFP+ epithelial cell": "Epithelial",
    "GATA4+ mesothelial cell": "Epithelial",
    "CRABP1+ mesothelial cell": "Epithelial",
    "Cycling mesothelial cell": "Epithelial",

    # -------------------------
    # Specialized
    # -------------------------
    "MT2A+ granulosa cell": "Specialized",
    "OSR1+ granulosa cell": "Specialized",
    "TAC1+ granulosa cell": "Specialized",
    "PAX8+ granulosa cell": "Specialized",
    "Retinoid acid responsive oocyte": "Specialized",
    "Mitotic oocyte": "Specialized",
    "Meiotic oocyte": "Specialized",
    "Oocyte": "Specialized",
    "Primordial germ cell": "Specialized",
    "Cycling primordial germ cell": "Specialized",
    "Endocrine cell": "Specialized",
    "Red blood cell": "Specialized",
    "Glial cell": "Specialized",

    # -------------------------
    # Unknown
    # -------------------------
    "Podocyte": "Unknown",  # not expected in ovary, so map to Unknown
    "Unknown": "Unknown",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
Fibroblast_Myofibroblast    56739
Specialized                 42578
Epithelial                  10283
Blood_vessel                 6791
T_NK                         4049
Myeloid                      2734
Unknown                       772
B_Plasma                      182
Melanocyte                      0
Name: count, dtype: int64


In [13]:
new_adata

AnnData object with n_obs × n_vars = 124128 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'age_group', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [14]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [15]:
report

{'n_input_genes': 33538,
 'n_mapped_genes': 23480,
 'pct_genes_mapped': 70.01013775418927,
 'n_output_genes_after_collapse': 23398,
 'n_ambiguous_genes': 16,
 'n_unmapped_or_no_ensembl': 10042,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 10058,
 'n_duplicate_ensembl_ids': 77,
 'n_duplicate_input_features_collapsed': 82,
 'total_counts_before': 1069486634.0,
 'total_counts_retained': 1058837966.0,
 'pct_total_counts_retained': 99.00431967436818,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [16]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
MIR1302-2HG,MIR1302-2HG,MIR1302-2HG,ENSG00000243485,approved_symbol
FAM138A,FAM138A,FAM138A,ENSG00000237613,approved_symbol
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
...,...,...,...,...
AC233755.2,AC233755.2,<NA>,<NA>,unmapped_or_no_ensembl
AC233755.1,AC233755.1,<NA>,<NA>,unmapped_or_no_ensembl
AC240274.1,AC240274.1,<NA>,<NA>,unmapped_or_no_ensembl
AC213203.1,AC213203.1,<NA>,<NA>,unmapped_or_no_ensembl


In [17]:
adata_ensembl

AnnData object with n_obs × n_vars = 124128 × 23398
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'age_group', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [18]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Pancreas

In [19]:
tissue = 'Pancreas'

In [20]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 188369 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'race', 'rna_source'
    var: 'gene'
    obsm: 'X_pca', 'X_umap'

In [21]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,anatomical_site,disease,platform,age_group,cell_sorting,subject_id,age,gender,race,rna_source
AAACCTGAGAACAATC-1--CRX030786,3499.0,1703,Vascular smooth muscle cell,CRX030786,CRA001160,control,pancreas,NA,control,10x3',NA,NA,NA,NaN,F,NA,cell
AAACCTGAGTATGACA-1--CRX030786,2363.0,1141,Capillary EC,CRX030786,CRA001160,control,pancreas,NA,control,10x3',NA,NA,NA,NaN,F,NA,cell
AAACCTGCAACCGCCA-1--CRX030786,9053.0,3358,Pancreatic ductal cell,CRX030786,CRA001160,control,pancreas,NA,control,10x3',NA,NA,NA,NaN,F,NA,cell
AAACCTGCAATCTACG-1--CRX030786,2369.0,1226,Capillary EC,CRX030786,CRA001160,control,pancreas,NA,control,10x3',NA,NA,NA,NaN,F,NA,cell
AAACCTGCATACGCCG-1--CRX030786,1692.0,958,Capillary EC,CRX030786,CRA001160,control,pancreas,NA,control,10x3',NA,NA,NA,NaN,F,NA,cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGAGCGATCGA-1--GSM6846538,1866.0,929,Pericyte,GSM6846538,GSE221156,control,pancreas,pancreatic islet,control,10x3',NA,NA,NA,60.0,F,Hispanic,cell
TTTGTTGAGGGACTGT-1--GSM6846538,17967.0,3843,Beta cell,GSM6846538,GSE221156,control,pancreas,pancreatic islet,control,10x3',NA,NA,NA,60.0,F,Hispanic,cell
TTTGTTGAGTGACCTT-1--GSM6846538,31052.0,4599,Beta cell,GSM6846538,GSE221156,control,pancreas,pancreatic islet,control,10x3',NA,NA,NA,60.0,F,Hispanic,cell
TTTGTTGGTCGAAACG-1--GSM6846538,13582.0,3872,Beta cell,GSM6846538,GSE221156,control,pancreas,pancreatic islet,control,10x3',NA,NA,NA,60.0,F,Hispanic,cell


In [22]:
adata.var

,gene
MIR1302-2HG,MIR1302-2HG
FAM138A,FAM138A
OR4F5,OR4F5
AL627309.1,AL627309.1
AL627309.3,AL627309.3
...,...
AC233755.2,AC233755.2
AC233755.1,AC233755.1
AC240274.1,AC240274.1
AC213203.1,AC213203.1


In [23]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [24]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 188369 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'race', 'rna_source', 'tissue_HistoST'
    var: 'gene_name'


In [25]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
Pancreatic ductal cell         39965
Alpha cell                     30012
Beta cell                      29078
Acinar cell                    20606
Capillary EC                   17847
Fibroblast                      6565
Delta cell                      6268
PP cell                         5690
KLRB1 CD8 T cell                5496
SPP1 macrophage                 4419
Memory B cell                   2541
Pericyte                        2170
GZMK CD8 T cell                 2090
Naive CD4 T cell                2043
Vascular smooth muscle cell     1864
CD14 monocyte                   1605
Memory CD4 T cell               1507
Venous EC                       1350
NK cell                         1328
Epsilon cell                     936
Neutrophil                       920
Mast cell                        704
Treg cell                        640
Plasma cell                      539
LYVE1 macrophage                 440
EndoMT                           418
Naive B cell       

In [26]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK
    # -------------------------
    "KLRB1 CD8 T cell": "T_NK",
    "GZMK CD8 T cell": "T_NK",
    "Naive CD4 T cell": "T_NK",
    "Memory CD4 T cell": "T_NK",
    "Treg cell": "T_NK",
    "NK cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "Memory B cell": "B_Plasma",
    "Naive B cell": "B_Plasma",
    "Germinal center B cell": "B_Plasma",
    "Plasma cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "SPP1 macrophage": "Myeloid",
    "LYVE1 macrophage": "Myeloid",
    "CD14 monocyte": "Myeloid",
    "CD16 monocyte": "Myeloid",
    "cDC1": "Myeloid",
    "cDC2": "Myeloid",
    "mregDC": "Myeloid",
    "pDC": "Myeloid",
    "Neutrophil": "Myeloid",
    "Mast cell": "Myeloid",
    "Basophil": "Myeloid",

    # -------------------------
    # Blood vessel
    # -------------------------
    "Capillary EC": "Blood_vessel",
    "Venous EC": "Blood_vessel",
    "Lymphatic EC": "Blood_vessel",
    "Pericyte": "Blood_vessel",
    "Vascular smooth muscle cell": "Blood_vessel",
    "EndoMT": "Blood_vessel",

    # -------------------------
    # Fibroblast / Myofibroblast
    # -------------------------
    "Fibroblast": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "Pancreatic ductal cell": "Epithelial",
    "Acinar cell": "Epithelial",

    # -------------------------
    # Specialized
    # -------------------------
    "Alpha cell": "Specialized",
    "Beta cell": "Specialized",
    "Delta cell": "Specialized",
    "PP cell": "Specialized",
    "Epsilon cell": "Specialized",
    "Schwann cell": "Specialized",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
Specialized                 72245
Epithelial                  60571
Blood_vessel                23729
T_NK                        13104
Myeloid                      8620
Fibroblast_Myofibroblast     6565
B_Plasma                     3535
Melanocyte                      0
Unknown                         0
Name: count, dtype: int64


In [27]:
new_adata

AnnData object with n_obs × n_vars = 188369 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'race', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [28]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [29]:
report

{'n_input_genes': 33538,
 'n_mapped_genes': 23480,
 'pct_genes_mapped': 70.01013775418927,
 'n_output_genes_after_collapse': 23398,
 'n_ambiguous_genes': 16,
 'n_unmapped_or_no_ensembl': 10042,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 10058,
 'n_duplicate_ensembl_ids': 77,
 'n_duplicate_input_features_collapsed': 82,
 'total_counts_before': 2802594698.0,
 'total_counts_retained': 2778675026.0,
 'pct_total_counts_retained': 99.14651690388662,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [30]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
MIR1302-2HG,MIR1302-2HG,MIR1302-2HG,ENSG00000243485,approved_symbol
FAM138A,FAM138A,FAM138A,ENSG00000237613,approved_symbol
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
...,...,...,...,...
AC233755.2,AC233755.2,<NA>,<NA>,unmapped_or_no_ensembl
AC233755.1,AC233755.1,<NA>,<NA>,unmapped_or_no_ensembl
AC240274.1,AC240274.1,<NA>,<NA>,unmapped_or_no_ensembl
AC213203.1,AC213203.1,<NA>,<NA>,unmapped_or_no_ensembl


In [31]:
adata_ensembl

AnnData object with n_obs × n_vars = 188369 × 23398
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'age_group', 'cell_sorting', 'subject_id', 'age', 'gender', 'race', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [32]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Prostate

In [33]:
tissue = 'Prostate'

In [34]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 21030 × 60606
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'scvi_leiden_res05_tissue', 'sample_number', 'suspension_type', 'tissue_type', 'disease_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'developm

In [35]:
adata.obs

,donor_id,tissue_in_publication,anatomical_position,method,cdna_plate,library_plate,notes,cdna_well,assay_ontology_term_id,sample_id,...,self_reported_ethnicity_ontology_term_id,development_stage_ontology_term_id,cell_type,assay,disease,sex,tissue,self_reported_ethnicity,development_stage,observation_joinid
TSP8_Prostate_NA_SS2_B134140_B134704_Epithelial_J19,TSP8,Prostate,NA,smartseq,B134140,B134704,Epithelial,J19,EFO:0008931,TSP8_Prostate_NA_SS2_B134140_B134704_Epithelial,...,HANCESTRO:0612,HsapDv:0000150,"CD8-positive, alpha-beta T cell",Smart-seq2,normal,male,prostate gland,Hispanic or Latin,56-year-old stage,vtl9%HT*-e
TSP8_Prostate_NA_SS2_B134140_B134704_Epithelial_L5,TSP8,Prostate,NA,smartseq,B134140,B134704,Epithelial,L5,EFO:0008931,TSP8_Prostate_NA_SS2_B134140_B134704_Epithelial,...,HANCESTRO:0612,HsapDv:0000150,"CD8-positive, alpha-beta T cell",Smart-seq2,normal,male,prostate gland,Hispanic or Latin,56-year-old stage,t&36VS0`}%
TSP8_Prostate_NA_SS2_B134140_B134704_Epithelial_N19,TSP8,Prostate,NA,smartseq,B134140,B134704,Epithelial,N19,EFO:0008931,TSP8_Prostate_NA_SS2_B134140_B134704_Epithelial,...,HANCESTRO:0612,HsapDv:0000150,luminal cell of prostate epithelium,Smart-seq2,normal,male,prostate gland,Hispanic or Latin,56-year-old stage,Jn7J%;=X#x
TSP8_Prostate_NA_SS2_B134140_B134704_Epithelial_G8,TSP8,Prostate,NA,smartseq,B134140,B134704,Epithelial,G8,EFO:0008931,TSP8_Prostate_NA_SS2_B134140_B134704_Epithelial,...,HANCESTRO:0612,HsapDv:0000150,B cell,Smart-seq2,normal,male,prostate gland,Hispanic or Latin,56-year-old stage,>R2qcT`q9(
TSP8_Prostate_NA_SS2_B134140_B134704_Epithelial_A10,TSP8,Prostate,NA,smartseq,B134140,B134704,Epithelial,A10,EFO:0008931,TSP8_Prostate_NA_SS2_B134140_B134704_Epithelial,...,HANCESTRO:0612,HsapDv:0000150,fibroblast,Smart-seq2,normal,male,prostate gland,Hispanic or Latin,56-year-old stage,kie`rdek@#
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TSP8_Prostate_NA_10X_1_1_TTTGTTGGTAAGCAAT,TSP8,Prostate,NA,10X,nan,nan,None,nan,EFO:0009922,TSP8_Prostate_NA_10X_1_1,...,HANCESTRO:0612,HsapDv:0000150,basal cell of prostate epithelium,10x 3' v3,normal,male,prostate gland,Hispanic or Latin,56-year-old stage,6bU~xot_`u
TSP8_Prostate_NA_10X_1_1_TTTGTTGGTGGCAGAT,TSP8,Prostate,NA,10X,nan,nan,None,nan,EFO:0009922,TSP8_Prostate_NA_10X_1_1,...,HANCESTRO:0612,HsapDv:0000150,luminal cell of prostate epithelium,10x 3' v3,normal,male,prostate gland,Hispanic or Latin,56-year-old stage,ZLcVtdthHR
TSP8_Prostate_NA_10X_1_1_TTTGTTGGTGGGTTGA,TSP8,Prostate,NA,10X,nan,nan,None,nan,EFO:0009922,TSP8_Prostate_NA_10X_1_1,...,HANCESTRO:0612,HsapDv:0000150,luminal cell of prostate epithelium,10x 3' v3,normal,male,prostate gland,Hispanic or Latin,56-year-old stage,&)>^~x=L6x
TSP8_Prostate_NA_10X_1_1_TTTGTTGTCACGATCA,TSP8,Prostate,NA,10X,nan,nan,None,nan,EFO:0009922,TSP8_Prostate_NA_10X_1_1,...,HANCESTRO:0612,HsapDv:0000150,luminal cell of prostate epithelium,10x 3' v3,normal,male,prostate gland,Hispanic or Latin,56-year-old stage,4tT#|TOKU@


In [36]:
adata.var

,ensembl_id,genome,mt,ercc,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,mean,std,feature_is_filtered,feature_name,feature_reference,feature_biotype,feature_length,feature_type
ENSG00000000003,ENSG00000000003.15,Gencode_v41,False,False,161872,2.379617,91.800375,4697694.0,0.092427,0.266598,False,TSPAN6,NCBITaxon:9606,gene,2396,protein_coding
ENSG00000000005,ENSG00000000005.6,Gencode_v41,False,False,9323,0.220273,99.527743,434850.0,0.000630,0.022016,False,TNMD,NCBITaxon:9606,gene,873,protein_coding
ENSG00000000419,ENSG00000000419.14,Gencode_v41,False,False,461590,3.523875,76.618161,6956619.0,0.289896,0.454315,False,DPM1,NCBITaxon:9606,gene,1262,protein_coding
ENSG00000000457,ENSG00000000457.14,Gencode_v41,False,False,156149,0.493041,92.090273,973332.0,0.097530,0.283859,False,SCYL3,NCBITaxon:9606,gene,2916,protein_coding
ENSG00000000460,ENSG00000000460.17,Gencode_v41,False,False,120250,0.281519,93.908737,555757.0,0.037517,0.183069,False,FIRRM,NCBITaxon:9606,gene,2661,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000290162,ENSG00000290162.1,Gencode_v41,False,False,600,0.001394,99.969607,2751.0,0.000016,0.002280,False,ENSG00000290162,NCBITaxon:9606,gene,149,unprocessed_pseudogene
ENSG00000290163,ENSG00000290163.1,Gencode_v41,False,False,1814,0.006923,99.908112,13667.0,0.000486,0.019340,False,ENSG00000290163,NCBITaxon:9606,gene,144,unprocessed_pseudogene
ENSG00000290164,ENSG00000290164.1,Gencode_v41,False,False,934,0.005787,99.952688,11424.0,0.000494,0.021458,False,ENSG00000290164,NCBITaxon:9606,gene,138,unprocessed_pseudogene
ENSG00000290165,ENSG00000290165.1,Gencode_v41,False,False,4283,0.014494,99.783045,28613.0,0.001415,0.035578,False,ENSG00000290165,NCBITaxon:9606,gene,144,unprocessed_pseudogene


In [37]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: False
No negative values: True
.X may NOT be raw counts


In [38]:
for layer in adata.layers.keys():
    X = adata.layers[layer]
    sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]
    
    print(f"\nLayer: {layer}")
    print("  min:", sample.min())
    print("  max:", sample.max())
    print("  integer-like:", np.allclose(sample, np.round(sample)))


Layer: decontXcounts
  min: 0
  max: 5233407
  integer-like: True

Layer: scale_data
  min: 0.0
  max: 145.01724
  integer-like: False


In [39]:
# Create new AnnData standardized

new_adata = adata.copy()

# Put counts in .X first
new_adata.X = new_adata.layers["decontXcounts"].copy()

# obs edits
new_adata.obs = new_adata.obs.copy()

# Rename cell_type to original_cell_type
new_adata.obs = new_adata.obs.rename(columns={"cell_type": "original_cell_type"})

# Rename total_counts as total_count
new_adata.obs = new_adata.obs.rename(columns={"total_counts": "total_count"})

# Tissue HistoST
new_adata.obs["tissue_HistoST"] = tissue

# var edits
new_adata.var = new_adata.var.copy()
new_adata.var = new_adata.var.rename(columns={"feature_name": "gene_name"})
new_adata.var["gene_name"] = new_adata.var["gene_name"].astype(str)
new_adata.var.index = new_adata.var["gene_name"]
new_adata.var.index.name = None

# drop extras
new_adata = sc.AnnData(
    X=new_adata.X.copy(),
    obs=new_adata.obs.copy(),
    var=new_adata.var.copy(),
)

new_adata

$HOME/miniforge3/envs/process_slide/lib/python3.12/site-packages/anndata/_core/anndata.py:1794: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 21030 × 60606
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_count', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'scvi_leiden_res05_tissue', 'sample_number', 'suspension_type', 'tissue_type', 'disease_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'original_cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', '

In [40]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
luminal cell of prostate epithelium    10652
basal cell of prostate epithelium       3495
CD8-positive, alpha-beta T cell         2516
endothelial cell                        1205
macrophage                              1022
fibroblast                               519
mature NK T cell                         487
smooth muscle cell                       477
neutrophil                               261
erythrocyte                              139
CD4-positive, alpha-beta T cell          116
B cell                                   101
mast cell                                 26
T cell                                    14
Name: count, dtype: int64

In [41]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK
    # -------------------------
    "CD8-positive, alpha-beta T cell": "T_NK",
    "CD4-positive, alpha-beta T cell": "T_NK",
    "mature NK T cell": "T_NK",
    "T cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "B cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "macrophage": "Myeloid",
    "neutrophil": "Myeloid",
    "mast cell": "Myeloid",

    # -------------------------
    # Blood vessel / mural
    # -------------------------
    "endothelial cell": "Blood_vessel",
    "smooth muscle cell": "Blood_vessel",

    # -------------------------
    # Fibroblast / Myofibroblast
    # -------------------------
    "fibroblast": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "luminal cell of prostate epithelium": "Epithelial",
    "basal cell of prostate epithelium": "Epithelial",

    # -------------------------
    # Specialized
    # -------------------------
    "erythrocyte": "Specialized",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
Epithelial                  14147
T_NK                         3133
Blood_vessel                 1682
Myeloid                      1309
Fibroblast_Myofibroblast      519
Specialized                   139
B_Plasma                      101
Melanocyte                      0
Unknown                         0
Name: count, dtype: int64


In [42]:
new_adata

AnnData object with n_obs × n_vars = 21030 × 60606
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_count', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'scvi_leiden_res05_tissue', 'sample_number', 'suspension_type', 'tissue_type', 'disease_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'original_cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', '

In [43]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [44]:
report

{'n_input_genes': 60606,
 'n_mapped_genes': 41134,
 'pct_genes_mapped': 67.87116787116787,
 'n_output_genes_after_collapse': 40992,
 'n_ambiguous_genes': 111,
 'n_unmapped_or_no_ensembl': 19361,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 19472,
 'n_duplicate_ensembl_ids': 63,
 'n_duplicate_input_features_collapsed': 142,
 'total_counts_before': 2929550765.0,
 'total_counts_retained': 2915907567.0,
 'pct_total_counts_retained': 99.53429043923737,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [45]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
TSPAN6,TSPAN6,TSPAN6,ENSG00000000003,approved_symbol
TNMD,TNMD,TNMD,ENSG00000000005,approved_symbol
DPM1,DPM1,DPM1,ENSG00000000419,approved_symbol
SCYL3,SCYL3,SCYL3,ENSG00000000457,approved_symbol
FIRRM,FIRRM,FIRRM,ENSG00000000460,approved_symbol
...,...,...,...,...
ENSG00000290162,ENSG00000290162,<NA>,<NA>,unmapped_or_no_ensembl
ENSG00000290163,ENSG00000290163,<NA>,<NA>,unmapped_or_no_ensembl
ENSG00000290164,ENSG00000290164,<NA>,<NA>,unmapped_or_no_ensembl
ENSG00000290165,ENSG00000290165,<NA>,<NA>,unmapped_or_no_ensembl


In [46]:
adata_ensembl

AnnData object with n_obs × n_vars = 21030 × 40992
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_count', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'scvi_leiden_res05_tissue', 'sample_number', 'suspension_type', 'tissue_type', 'disease_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'original_cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', '

In [47]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Skin

In [48]:
tissue = 'Skin'

In [49]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

$HOME/miniforge3/envs/process_slide/lib/python3.12/site-packages/anndata/_io/h5ad.py:266: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  return AnnData(**{


AnnData object with n_obs × n_vars = 394594 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'treatment', 'subject_id', 'age', 'gender', 'disease_stage', 'rna_source'
    var: 'gene'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances'

In [50]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,anatomical_site,disease,platform,cell_sorting,treatment,subject_id,age,gender,disease_stage,rna_source
AAACCCAAGAACAGGA-1--GSM4994379,8785.0,2334,Venous EC,GSM4994379,GSE163973,disease tissue (non-cancer),skin,NA,keloid,10x3',NA,NA,NA,NaN,NA,NA,cell
AAACCCAAGATCCTAC-1--GSM4994379,15858.0,3588,CFD+MGP+ fibroblast,GSM4994379,GSE163973,disease tissue (non-cancer),skin,NA,keloid,10x3',NA,NA,NA,NaN,NA,NA,cell
AAACCCAAGCCTGACC-1--GSM4994379,12628.0,2984,Venous EC,GSM4994379,GSE163973,disease tissue (non-cancer),skin,NA,keloid,10x3',NA,NA,NA,NaN,NA,NA,cell
AAACCCAAGCGCAATG-1--GSM4994379,7064.0,1926,Venous EC,GSM4994379,GSE163973,disease tissue (non-cancer),skin,NA,keloid,10x3',NA,NA,NA,NaN,NA,NA,cell
AAACCCACAAACTCGT-1--GSM4994379,5225.0,1992,Venous EC,GSM4994379,GSE163973,disease tissue (non-cancer),skin,NA,keloid,10x3',NA,NA,NA,NaN,NA,NA,cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTAGTCCCGAC-1--GSM4567898,3589.0,1342,Treg cell,GSM4567898,GSE151177,disease tissue (non-cancer),skin,NA,psoriasis,10x3',NA,NA,NA,NaN,NA,NA,cell
TTTGGTTCATCTATCT-1--GSM4567898,5218.0,1537,Granular keratinocyte,GSM4567898,GSE151177,disease tissue (non-cancer),skin,NA,psoriasis,10x3',NA,NA,NA,NaN,NA,NA,cell
TTTGGTTGTCCTACGG-1--GSM4567898,4677.0,1581,CD4 T cell,GSM4567898,GSE151177,disease tissue (non-cancer),skin,NA,psoriasis,10x3',NA,NA,NA,NaN,NA,NA,cell
TTTGGTTTCCGTAATG-1--GSM4567898,1167.0,514,Granular keratinocyte,GSM4567898,GSE151177,disease tissue (non-cancer),skin,NA,psoriasis,10x3',NA,NA,NA,NaN,NA,NA,cell


In [51]:
adata.var

,gene
MIR1302-2HG,MIR1302-2HG
FAM138A,FAM138A
OR4F5,OR4F5
AL627309.1,AL627309.1
AL627309.3,AL627309.3
...,...
AC233755.2,AC233755.2
AC233755.1,AC233755.1
AC240274.1,AC240274.1
AC213203.1,AC213203.1


In [52]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [53]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 394594 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'treatment', 'subject_id', 'age', 'gender', 'disease_stage', 'rna_source', 'tissue_HistoST'
    var: 'gene_name'


In [54]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
Basal keratinocyte             60921
Spinous keratinocyte           42995
CD8 T cell                     30344
Venous EC                      24958
CD4 T cell                     24284
CFD+MGP+ fibroblast            22358
Merkel cell                    15456
cDC2                           13243
Pericyte                       12526
Monocyte                       12239
Tfh cell                       10173
Treg cell                       9932
Vascular smooth muscle cell     9179
Granular keratinocyte           7972
CCL19+APOE+ fibroblast          7944
Cycling keratinocyte            7930
CD9+APCDD1+ fibroblast          7878
APOD+PTGDS+ fibroblast          7812
Capillary EC                    6794
Memory B cell                   6402
Macrophage                      6209
Melanocyte                      5291
NK cell                         5119
ADAM12+ fibroblast              4619
Tip EC                          3679
Arterial EC                     3341
Lymphatic EC       

In [55]:
# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK
    # -------------------------
    "CD8 T cell": "T_NK",
    "CD4 T cell": "T_NK",
    "Tfh cell": "T_NK",
    "Treg cell": "T_NK",
    "NK cell": "T_NK",
    "CXCL13 exhausted CD8 T cell": "T_NK",
    "Cycling T/NK cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "Memory B cell": "B_Plasma",
    "Naive B cell": "B_Plasma",
    "Plasma cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "Monocyte": "Myeloid",
    "Macrophage": "Myeloid",
    "cDC1": "Myeloid",
    "cDC2": "Myeloid",
    "mregDC": "Myeloid",
    "pDC": "Myeloid",
    "Mast cell": "Myeloid",

    # -------------------------
    # Blood vessel
    # -------------------------
    "Venous EC": "Blood_vessel",
    "Capillary EC": "Blood_vessel",
    "Tip EC": "Blood_vessel",
    "Arterial EC": "Blood_vessel",
    "Lymphatic EC": "Blood_vessel",
    "Pericyte": "Blood_vessel",
    "Vascular smooth muscle cell": "Blood_vessel",

    # -------------------------
    # Fibroblast / Myofibroblast
    # -------------------------
    "CFD+MGP+ fibroblast": "Fibroblast_Myofibroblast",
    "CCL19+APOE+ fibroblast": "Fibroblast_Myofibroblast",
    "CD9+APCDD1+ fibroblast": "Fibroblast_Myofibroblast",
    "APOD+PTGDS+ fibroblast": "Fibroblast_Myofibroblast",
    "ADAM12+ fibroblast": "Fibroblast_Myofibroblast",
    "FGFBP2+OLFML2A+ fibroblast": "Fibroblast_Myofibroblast",
    "COCH+TNN+ fibroblast": "Fibroblast_Myofibroblast",
    "G2/M phase GPC3+ fibroblast": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "Basal keratinocyte": "Epithelial",
    "Spinous keratinocyte": "Epithelial",
    "Granular keratinocyte": "Epithelial",
    "Cycling keratinocyte": "Epithelial",
    "Secretory cell": "Epithelial",
    "Merkel cell": "Epithelial",

    # -------------------------
    # Specialized
    # -------------------------
    "Glial cell": "Specialized",

    # -------------------------
    # Melanocyte
    # -------------------------
    "Melanocyte": "Melanocyte",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
Epithelial                  137910
T_NK                         83731
Blood_vessel                 63748
Fibroblast_Myofibroblast     54269
Myeloid                      38128
B_Plasma                     10800
Melanocyte                    5291
Specialized                    717
Unknown                          0
Name: count, dtype: int64


In [56]:
new_adata

AnnData object with n_obs × n_vars = 394594 × 33538
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'treatment', 'subject_id', 'age', 'gender', 'disease_stage', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [57]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [58]:
report

{'n_input_genes': 33538,
 'n_mapped_genes': 23480,
 'pct_genes_mapped': 70.01013775418927,
 'n_output_genes_after_collapse': 23398,
 'n_ambiguous_genes': 16,
 'n_unmapped_or_no_ensembl': 10042,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 10058,
 'n_duplicate_ensembl_ids': 77,
 'n_duplicate_input_features_collapsed': 82,
 'total_counts_before': 2403843972.0,
 'total_counts_retained': 2392909364.0,
 'pct_total_counts_retained': 99.54511989432898,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [59]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
MIR1302-2HG,MIR1302-2HG,MIR1302-2HG,ENSG00000243485,approved_symbol
FAM138A,FAM138A,FAM138A,ENSG00000237613,approved_symbol
OR4F5,OR4F5,OR4F5,ENSG00000186092,approved_symbol
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
...,...,...,...,...
AC233755.2,AC233755.2,<NA>,<NA>,unmapped_or_no_ensembl
AC233755.1,AC233755.1,<NA>,<NA>,unmapped_or_no_ensembl
AC240274.1,AC240274.1,<NA>,<NA>,unmapped_or_no_ensembl
AC213203.1,AC213203.1,<NA>,<NA>,unmapped_or_no_ensembl


In [60]:
adata_ensembl

AnnData object with n_obs × n_vars = 394594 × 23398
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'anatomical_site', 'disease', 'platform', 'cell_sorting', 'treatment', 'subject_id', 'age', 'gender', 'disease_stage', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [61]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Tonsil

In [5]:
tissue = 'Tonsil'

In [6]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 273755 × 32577
    obs: 'nCount_RNA', 'nFeature_RNA', 'cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source'
    var: 'gene'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'

In [7]:
adata.obs

,nCount_RNA,nFeature_RNA,cell_type,sample_id,project_id,sample_type,tissue,disease,platform,cell_sorting,subject_id,age,gender,rna_source
wf4su8ny_h4yj8bv7_AAACGAATCCATTGGA-1--BCLL_13_T_1_hashing,1398.0,694,Macrophage,BCLL_13_T_1_hashing,EGAS00001006375,control,tonsil,control,10x3',NA,BCLL-13-T,5.0,F,cell
wf4su8ny_h4yj8bv7_ACGGTTAGTACGTACT-1--BCLL_13_T_1_hashing,1305.0,668,Macrophage,BCLL_13_T_1_hashing,EGAS00001006375,control,tonsil,control,10x3',NA,BCLL-13-T,5.0,F,cell
wf4su8ny_h4yj8bv7_ACTACGAGTTTCACTT-1--BCLL_13_T_1_hashing,1705.0,757,Macrophage,BCLL_13_T_1_hashing,EGAS00001006375,control,tonsil,control,10x3',NA,BCLL-13-T,5.0,F,cell
wf4su8ny_h4yj8bv7_AGAAGCGAGTGCCAGA-1--BCLL_13_T_1_hashing,2634.0,1125,Macrophage,BCLL_13_T_1_hashing,EGAS00001006375,control,tonsil,control,10x3',NA,BCLL-13-T,5.0,F,cell
wf4su8ny_h4yj8bv7_AGCCACGGTGGTAATA-1--BCLL_13_T_1_hashing,3462.0,782,mregDC,BCLL_13_T_1_hashing,EGAS00001006375,control,tonsil,control,10x3',NA,BCLL-13-T,5.0,F,cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCAAGATAGTCA-1--GSM3375767,8965.0,2475,MHCII high CD14 monocyte,GSM3375767,GSE119506,control,tonsil,control,10x3',CD14+ cells,NA,NaN,NA,cell
TTTGTCAAGTTCCACA-1--GSM3375767,7114.0,2007,MHCII high CD14 monocyte,GSM3375767,GSE119506,control,tonsil,control,10x3',CD14+ cells,NA,NaN,NA,cell
TTTGTCACAGTAACGG-1--GSM3375767,5114.0,1700,MHCII high CD14 monocyte,GSM3375767,GSE119506,control,tonsil,control,10x3',CD14+ cells,NA,NaN,NA,cell
TTTGTCAGTCTAACGT-1--GSM3375767,4661.0,1645,MHCII high CD14 monocyte,GSM3375767,GSE119506,control,tonsil,control,10x3',CD14+ cells,NA,NaN,NA,cell


In [8]:
adata.var

,gene
AL627309.1,AL627309.1
AL627309.3,AL627309.3
AL627309.4,AL627309.4
AL669831.2,AL669831.2
FAM87B,FAM87B
...,...
AC171558.1,AC171558.1
AC133551.1,AC133551.1
AC136612.1,AC136612.1
AC136616.1,AC136616.1


In [9]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: True
No negative values: True
.X likely contains raw counts


In [10]:
# Create new AnnData standardized

new_adata = sc.AnnData(
    X=adata.X.copy(),                     # keep X as is
    obs=adata.obs.copy(),
    var=adata.var.copy()
)

# Remove unwanted slots
new_adata.uns = {}
new_adata.obsm = {}
new_adata.varm = {}
new_adata.obsp = {}

# Modify obs
new_adata.obs = new_adata.obs.rename(columns={
    "nCount_RNA": "total_count",
    "cell_type": "original_cell_type"
})

# For obs.original_cell_type, fill NaNs with "Unknown"
new_adata.obs["original_cell_type"] = (
    new_adata.obs["original_cell_type"].fillna("Unknown")
)

new_adata.obs["tissue_HistoST"] = tissue

# Modify var
new_adata.var = new_adata.var.rename(columns={
    "gene": "gene_name"
})

print(new_adata)

AnnData object with n_obs × n_vars = 273755 × 32577
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST'
    var: 'gene_name'


In [11]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
Naive B cell                         44781
DZ/LZ B cell                         22913
Unswitched memory B cell             17194
Tfh cell                             16723
Class-switched memory B cell         15635
Memory CD4 T cell                    14767
Non-cycling GC B cell                14341
Naive CD4 T cell                     12791
Light zone GC B cell                 12640
FCRL4 memory B cell                  12567
Committed GC B cell                  10402
Treg cell                            10211
Histone high S phase DZ GC B cell     9343
S phase DZ GC B cell                  9288
Plasma cell                           8117
PIF1+ G2/M phase DZ GC B cell         7274
Memory CD8 T cell                     6046
Memory B cell precursor               4538
Pre-GC B cell                         4316
Naive CD8 T cell                      4213
G2/M phase DZ GC B cell               4049
MHCII high CD14 monocyte              2632
Plasma cell precursor              

In [12]:
# ct_1 mapping

# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK
    # -------------------------
    "Tfh cell": "T_NK",
    "Memory CD4 T cell": "T_NK",
    "Naive CD4 T cell": "T_NK",
    "Treg cell": "T_NK",
    "Memory CD8 T cell": "T_NK",
    "Naive CD8 T cell": "T_NK",
    "NK cell": "T_NK",
    "Double negative T cell": "T_NK",
    "Cycling T cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "Naive B cell": "B_Plasma",
    "DZ/LZ B cell": "B_Plasma",
    "Unswitched memory B cell": "B_Plasma",
    "Class-switched memory B cell": "B_Plasma",
    "Non-cycling GC B cell": "B_Plasma",
    "Light zone GC B cell": "B_Plasma",
    "FCRL4 memory B cell": "B_Plasma",
    "Committed GC B cell": "B_Plasma",
    "Histone high S phase DZ GC B cell": "B_Plasma",
    "S phase DZ GC B cell": "B_Plasma",
    "Plasma cell": "B_Plasma",
    "PIF1+ G2/M phase DZ GC B cell": "B_Plasma",
    "Memory B cell precursor": "B_Plasma",
    "Pre-GC B cell": "B_Plasma",
    "G2/M phase DZ GC B cell": "B_Plasma",
    "Plasma cell precursor": "B_Plasma",
    "INF-activated naive B cell": "B_Plasma",
    "PreB cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "MHCII high CD14 monocyte": "Myeloid",
    "Macrophage": "Myeloid",
    "pDC": "Myeloid",
    "cDC2": "Myeloid",
    "mregDC": "Myeloid",
    "Granulocyte": "Myeloid",
    "Light zone follicular DC": "Myeloid",
    "Dark zone follicular DC": "Myeloid",

    # -------------------------
    # Epithelial
    # -------------------------
    "Tonsillar crypt epithelial cell": "Epithelial",
    "Epithelial cell": "Epithelial",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
B_Plasma                    201142
T_NK                         66581
Myeloid                       5785
Epithelial                     247
Blood_vessel                     0
Fibroblast_Myofibroblast         0
Specialized                      0
Melanocyte                       0
Unknown                          0
Name: count, dtype: int64


In [13]:
new_adata

AnnData object with n_obs × n_vars = 273755 × 32577
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_name'

In [14]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [15]:
report

{'n_input_genes': 32577,
 'n_mapped_genes': 23361,
 'pct_genes_mapped': 71.71010221935721,
 'n_output_genes_after_collapse': 23284,
 'n_ambiguous_genes': 15,
 'n_unmapped_or_no_ensembl': 9201,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 9216,
 'n_duplicate_ensembl_ids': 73,
 'n_duplicate_input_features_collapsed': 77,
 'total_counts_before': 1498933174.0,
 'total_counts_retained': 1490047146.0,
 'pct_total_counts_retained': 99.4071765069895,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [16]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
AL627309.1,AL627309.1,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.3,AL627309.3,<NA>,<NA>,unmapped_or_no_ensembl
AL627309.4,AL627309.4,<NA>,<NA>,unmapped_or_no_ensembl
AL669831.2,AL669831.2,<NA>,<NA>,unmapped_or_no_ensembl
FAM87B,FAM87B,FAM87B,ENSG00000177757,approved_symbol
...,...,...,...,...
AC171558.1,AC171558.1,<NA>,<NA>,unmapped_or_no_ensembl
AC133551.1,AC133551.1,<NA>,<NA>,unmapped_or_no_ensembl
AC136612.1,AC136612.1,<NA>,<NA>,unmapped_or_no_ensembl
AC136616.1,AC136616.1,<NA>,<NA>,unmapped_or_no_ensembl


In [17]:
adata_ensembl

AnnData object with n_obs × n_vars = 273755 × 23284
    obs: 'total_count', 'nFeature_RNA', 'original_cell_type', 'sample_id', 'project_id', 'sample_type', 'tissue', 'disease', 'platform', 'cell_sorting', 'subject_id', 'age', 'gender', 'rna_source', 'tissue_HistoST', 'ct_1_cell_type'
    var: 'gene_ids', 'gene_name', 'n_input_features_collapsed', 'collapsed_from'
    uns: 'gene_id_mapping_report', 'gene_id_mapping_audit'

In [18]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])

## Uterus

In [19]:
tissue = 'Uterus'

In [20]:
adata = sc.read_h5ad(ref_folder_before / filename_dict[tissue])
adata

AnnData object with n_obs × n_vars = 22029 × 60606
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'scvi_leiden_res05_tissue', 'sample_number', 'suspension_type', 'tissue_type', 'disease_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'developm

In [21]:
adata.obs

,donor_id,tissue_in_publication,anatomical_position,method,cdna_plate,library_plate,notes,cdna_well,assay_ontology_term_id,sample_id,...,self_reported_ethnicity_ontology_term_id,development_stage_ontology_term_id,cell_type,assay,disease,sex,tissue,self_reported_ethnicity,development_stage,observation_joinid
TSP4_Uterus_NA_SS2_B112801_B134701_Epithelial_B3,TSP4,Uterus,NA,smartseq,B112801,B134701,Epithelial,B3,EFO:0008931,TSP4_Uterus_NA_SS2_B112801_B134701_Epithelial,...,unknown,HsapDv:0000132,epithelial cell of uterus,Smart-seq2,normal,female,uterus,unknown,38-year-old stage,6>mz+VgQYL
TSP4_Uterus_NA_SS2_B112801_B134701_Epithelial_B23,TSP4,Uterus,NA,smartseq,B112801,B134701,Epithelial,B23,EFO:0008931,TSP4_Uterus_NA_SS2_B112801_B134701_Epithelial,...,unknown,HsapDv:0000132,fibroblast,Smart-seq2,normal,female,uterus,unknown,38-year-old stage,k^IoxcYHWu
TSP4_Uterus_NA_SS2_B112801_B134701_Epithelial_M8,TSP4,Uterus,NA,smartseq,B112801,B134701,Epithelial,M8,EFO:0008931,TSP4_Uterus_NA_SS2_B112801_B134701_Epithelial,...,unknown,HsapDv:0000132,endothelial cell,Smart-seq2,normal,female,uterus,unknown,38-year-old stage,<B>R$ppLNc
TSP4_Uterus_NA_SS2_B112801_B134701_Epithelial_F23,TSP4,Uterus,NA,smartseq,B112801,B134701,Epithelial,F23,EFO:0008931,TSP4_Uterus_NA_SS2_B112801_B134701_Epithelial,...,unknown,HsapDv:0000132,epithelial cell of uterus,Smart-seq2,normal,female,uterus,unknown,38-year-old stage,nT)ye8^;q|
TSP4_Uterus_NA_SS2_B112801_B134701_Epithelial_L7,TSP4,Uterus,NA,smartseq,B112801,B134701,Epithelial,L7,EFO:0008931,TSP4_Uterus_NA_SS2_B112801_B134701_Epithelial,...,unknown,HsapDv:0000132,epithelial cell of uterus,Smart-seq2,normal,female,uterus,unknown,38-year-old stage,lyY-ys(1+4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TSP4_Uterus_Endometrium_10X_1_1_TTTGTTGAGCAGGGAG,TSP4,Uterus,Endometrium,10X,nan,nan,None,nan,EFO:0009922,TSP4_Uterus_Endometrium_10X_1_1,...,unknown,HsapDv:0000132,fibroblast,10x 3' v3,normal,female,endometrium,unknown,38-year-old stage,c68epN^#;!
TSP4_Uterus_Endometrium_10X_1_1_TTTGTTGAGTAGAATC,TSP4,Uterus,Endometrium,10X,nan,nan,None,nan,EFO:0009922,TSP4_Uterus_Endometrium_10X_1_1,...,unknown,HsapDv:0000132,"CD8-positive, alpha-beta T cell",10x 3' v3,normal,female,endometrium,unknown,38-year-old stage,;fx2j=c>z?
TSP4_Uterus_Endometrium_10X_1_1_TTTGTTGCAGCGTGCT,TSP4,Uterus,Endometrium,10X,nan,nan,None,nan,EFO:0009922,TSP4_Uterus_Endometrium_10X_1_1,...,unknown,HsapDv:0000132,fibroblast,10x 3' v3,normal,female,endometrium,unknown,38-year-old stage,;^9emno-ay
TSP4_Uterus_Endometrium_10X_1_1_TTTGTTGGTAGAATAC,TSP4,Uterus,Endometrium,10X,nan,nan,None,nan,EFO:0009922,TSP4_Uterus_Endometrium_10X_1_1,...,unknown,HsapDv:0000132,fibroblast,10x 3' v3,normal,female,endometrium,unknown,38-year-old stage,#tUk~N2vQt


In [22]:
adata.var

,ensembl_id,genome,mt,ercc,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,mean,std,feature_is_filtered,feature_name,feature_reference,feature_biotype,feature_length,feature_type
ENSG00000000003,ENSG00000000003.15,Gencode_v41,False,False,161872,2.379617,91.800375,4697694.0,0.067797,0.234980,False,TSPAN6,NCBITaxon:9606,gene,2396,protein_coding
ENSG00000000005,ENSG00000000005.6,Gencode_v41,False,False,9323,0.220273,99.527743,434850.0,0.001346,0.033168,False,TNMD,NCBITaxon:9606,gene,873,protein_coding
ENSG00000000419,ENSG00000000419.14,Gencode_v41,False,False,461590,3.523875,76.618161,6956619.0,0.292036,0.457286,False,DPM1,NCBITaxon:9606,gene,1262,protein_coding
ENSG00000000457,ENSG00000000457.14,Gencode_v41,False,False,156149,0.493041,92.090273,973332.0,0.052640,0.215329,False,SCYL3,NCBITaxon:9606,gene,2916,protein_coding
ENSG00000000460,ENSG00000000460.17,Gencode_v41,False,False,120250,0.281519,93.908737,555757.0,0.045223,0.205693,False,FIRRM,NCBITaxon:9606,gene,2661,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000290162,ENSG00000290162.1,Gencode_v41,False,False,600,0.001394,99.969607,2751.0,0.000049,0.005242,False,ENSG00000290162,NCBITaxon:9606,gene,149,unprocessed_pseudogene
ENSG00000290163,ENSG00000290163.1,Gencode_v41,False,False,1814,0.006923,99.908112,13667.0,0.000418,0.019165,False,ENSG00000290163,NCBITaxon:9606,gene,144,unprocessed_pseudogene
ENSG00000290164,ENSG00000290164.1,Gencode_v41,False,False,934,0.005787,99.952688,11424.0,0.000102,0.007770,False,ENSG00000290164,NCBITaxon:9606,gene,138,unprocessed_pseudogene
ENSG00000290165,ENSG00000290165.1,Gencode_v41,False,False,4283,0.014494,99.783045,28613.0,0.000543,0.020543,False,ENSG00000290165,NCBITaxon:9606,gene,144,unprocessed_pseudogene


In [23]:
# Check if .X looks like counts
X = adata.X

# If sparse, convert a small sample to dense for checking
sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]

is_integer_like = np.allclose(sample, np.round(sample))
has_no_negatives = np.min(sample) >= 0

print("Integer-like values:", is_integer_like)
print("No negative values:", has_no_negatives)

if is_integer_like and has_no_negatives:
    print(".X likely contains raw counts")
else:
    print(".X may NOT be raw counts")

Integer-like values: False
No negative values: True
.X may NOT be raw counts


In [24]:
for layer in adata.layers.keys():
    X = adata.layers[layer]
    sample = X[:1000].toarray() if hasattr(X, "toarray") else X[:1000]
    
    print(f"\nLayer: {layer}")
    print("  min:", sample.min())
    print("  max:", sample.max())
    print("  integer-like:", np.allclose(sample, np.round(sample)))


Layer: decontXcounts
  min: 0
  max: 5169181
  integer-like: True

Layer: scale_data
  min: 0.0
  max: 148.42169
  integer-like: False


In [25]:
# Create new AnnData standardized

new_adata = adata.copy()

# Put counts in .X first
new_adata.X = new_adata.layers["decontXcounts"].copy()

# obs edits
new_adata.obs = new_adata.obs.copy()

# Rename cell_type to original_cell_type
new_adata.obs = new_adata.obs.rename(columns={"cell_type": "original_cell_type"})

# Rename total_counts as total_count
new_adata.obs = new_adata.obs.rename(columns={"total_counts": "total_count"})

# Tissue HistoST
new_adata.obs["tissue_HistoST"] = tissue

# var edits
new_adata.var = new_adata.var.copy()
new_adata.var = new_adata.var.rename(columns={"feature_name": "gene_name"})
new_adata.var["gene_name"] = new_adata.var["gene_name"].astype(str)
new_adata.var.index = new_adata.var["gene_name"]
new_adata.var.index.name = None

# drop extras
new_adata = sc.AnnData(
    X=new_adata.X.copy(),
    obs=new_adata.obs.copy(),
    var=new_adata.var.copy(),
)

new_adata

$HOME/miniforge3/envs/process_slide/lib/python3.12/site-packages/anndata/_core/anndata.py:1794: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 22029 × 60606
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_count', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'scvi_leiden_res05_tissue', 'sample_number', 'suspension_type', 'tissue_type', 'disease_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'original_cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', '

In [26]:
new_adata.obs.original_cell_type.value_counts(dropna=False)

original_cell_type
fibroblast                                6593
endothelial cell                          4697
vascular associated smooth muscle cell    3595
epithelial cell of uterus                 1417
CD8-positive, alpha-beta T cell           1341
pericyte                                  1033
macrophage                                 929
myometrial cell                            917
CD4-positive, alpha-beta T cell            438
ciliated epithelial cell                   320
mature NK T cell                           207
neutrophil                                 103
myofibroblast cell                         101
natural killer cell                         92
mast cell                                   65
B cell                                      50
T cell                                      47
monocyte                                    21
plasma cell                                 19
skeletal muscle satellite stem cell         18
endothelial cell of lymphatic vessel     

In [27]:
# ct_1 mapping

# ct_1 mapping

ct_1_map = {
    # -------------------------
    # T / NK / ILC
    # -------------------------
    "CD8-positive, alpha-beta T cell": "T_NK",
    "CD4-positive, alpha-beta T cell": "T_NK",
    "T cell": "T_NK",
    "regulatory T cell": "T_NK",
    "natural killer cell": "T_NK",
    "mature NK T cell": "T_NK",
    "innate lymphoid cell": "T_NK",

    # -------------------------
    # B / Plasma
    # -------------------------
    "B cell": "B_Plasma",
    "plasma cell": "B_Plasma",

    # -------------------------
    # Myeloid
    # -------------------------
    "macrophage": "Myeloid",
    "monocyte": "Myeloid",
    "intermediate monocyte": "Myeloid",
    "myeloid dendritic cell": "Myeloid",
    "neutrophil": "Myeloid",
    "granulocyte": "Myeloid",
    "mast cell": "Myeloid",

    # -------------------------
    # Blood vessel / vascular-associated
    # -------------------------
    "endothelial cell": "Blood_vessel",
    "endothelial cell of lymphatic vessel": "Blood_vessel",
    "pericyte": "Blood_vessel",
    "vascular associated smooth muscle cell": "Blood_vessel",
    "adventitial cell": "Blood_vessel",
    "myometrial cell": "Blood_vessel",

    # -------------------------
    # Fibroblast / Myofibroblast
    # -------------------------
    "fibroblast": "Fibroblast_Myofibroblast",
    "myofibroblast cell": "Fibroblast_Myofibroblast",

    # -------------------------
    # Epithelial
    # -------------------------
    "epithelial cell of uterus": "Epithelial",
    "ciliated epithelial cell": "Epithelial",
    "epithelial fate stem cell": "Epithelial",

    # -------------------------
    # Specialized
    # -------------------------
    "skeletal muscle satellite stem cell": "Specialized",
}

# Create the new column
new_adata.obs["ct_1_cell_type"] = (
    new_adata.obs["original_cell_type"]
    .map(ct_1_map)
    .fillna("Unknown")
)

# Categorical
ct_1_order = [
    "T_NK",
    "B_Plasma",
    "Myeloid",
    "Blood_vessel",
    "Fibroblast_Myofibroblast",
    "Epithelial",
    "Specialized",
    "Melanocyte",
    "Unknown",
]

new_adata.obs["ct_1_cell_type"] = pd.Categorical(
    new_adata.obs["ct_1_cell_type"],
    categories=ct_1_order,
    ordered=False
)

# Sanity check: see whether any labels were not explicitly mapped
unmapped = sorted(
    set(new_adata.obs["original_cell_type"].astype(str).unique()) - set(ct_1_map.keys())
)
print("Unmapped labels:", unmapped)

# Check result
print(new_adata.obs["ct_1_cell_type"].value_counts(dropna=False))

Unmapped labels: []
ct_1_cell_type
Blood_vessel                10252
Fibroblast_Myofibroblast     6694
T_NK                         2131
Epithelial                   1739
Myeloid                      1126
B_Plasma                       69
Specialized                    18
Melanocyte                      0
Unknown                         0
Name: count, dtype: int64


In [28]:
new_adata

AnnData object with n_obs × n_vars = 22029 × 60606
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_count', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'scvi_leiden_res05_tissue', 'sample_number', 'suspension_type', 'tissue_type', 'disease_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'original_cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', '

In [29]:
adata_ensembl, report, audit = map_human_gene_symbols_to_ensembl(new_adata)

In [30]:
report

{'n_input_genes': 60606,
 'n_mapped_genes': 41134,
 'pct_genes_mapped': 67.87116787116787,
 'n_output_genes_after_collapse': 40992,
 'n_ambiguous_genes': 111,
 'n_unmapped_or_no_ensembl': 19361,
 'n_empty_gene_names': 0,
 'n_dropped_genes_total': 19472,
 'n_duplicate_ensembl_ids': 63,
 'n_duplicate_input_features_collapsed': 142,
 'total_counts_before': 2235373204.0,
 'total_counts_retained': 2219720294.0,
 'pct_total_counts_retained': 99.29976301174271,
 'hgnc_url': 'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}

In [31]:
audit

,input_symbol,resolved_symbol,gene_ids,mapping_status
TSPAN6,TSPAN6,TSPAN6,ENSG00000000003,approved_symbol
TNMD,TNMD,TNMD,ENSG00000000005,approved_symbol
DPM1,DPM1,DPM1,ENSG00000000419,approved_symbol
SCYL3,SCYL3,SCYL3,ENSG00000000457,approved_symbol
FIRRM,FIRRM,FIRRM,ENSG00000000460,approved_symbol
...,...,...,...,...
ENSG00000290162,ENSG00000290162,<NA>,<NA>,unmapped_or_no_ensembl
ENSG00000290163,ENSG00000290163,<NA>,<NA>,unmapped_or_no_ensembl
ENSG00000290164,ENSG00000290164,<NA>,<NA>,unmapped_or_no_ensembl
ENSG00000290165,ENSG00000290165,<NA>,<NA>,unmapped_or_no_ensembl


In [32]:
adata_ensembl

AnnData object with n_obs × n_vars = 22029 × 40992
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_count', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'scvi_leiden_res05_tissue', 'sample_number', 'suspension_type', 'tissue_type', 'disease_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'original_cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', '

In [33]:
audit = adata_ensembl.uns["gene_id_mapping_audit"].copy()

for col in audit.columns:
    if pd.api.types.is_object_dtype(audit[col]) or pd.api.types.is_string_dtype(audit[col]):
        audit[col] = audit[col].where(pd.notna(audit[col]), "")
        audit[col] = audit[col].astype(str)

adata_ensembl.uns["gene_id_mapping_audit"] = audit

adata_ensembl.write_h5ad(ref_folder_after / filename_dict[tissue])